# UNIFESP v2 -- stage 2: train on ALL labelled images (+ optional pseudo-labels) to push the Kaggle score


**Attach:** `felipekitamura/unifesp-xray-bodypart-classification`, **GPU T4 x2**, Internet on. One account, one notebook at a time.

* **2a** -- the best configs of stage 1 (ConvNeXt-T, ResNet50-heavy, EffNet-B3-heavy, EffNetV2-S, DenseNet121; 2 seeds for the first three) trained on **all 1,738 labelled images**
  in 5 patient-level folds (+25 % data vs stage 1, which kept 323 images aside). Ensemble weights / thresholds from OOF only -> `submission_s2a.csv`.
* **Pseudo-label round** -- confident predictions of the 2a ensemble on the 743 unlabelled test images become extra training targets (the competition rules state
  pseudo-labelling is allowed; no test label is ever touched). **2b** retrains the same configs with them -> `submission_s2b.csv`; **s2u** = ensemble of 2a + 2b.
* UNMEASURED time: 40 jobs per round, ~2.5 min each wall => ~100 min per round, ~3.5-4 h in total.

Honesty notes for the paper: there is NO held-out set here (everything is used for training), so the only internal number is OOF, which for the pseudo-label round is slightly optimistic;
judge the rounds by the **Late-Submission score** and report ALL submissions you made (not just the best), and say that 2b/s2u are transductive (use unlabelled test images).
The paper's primary numbers stay those of stage 1 (held-out set, CIs).

In [ ]:
import os
images_root = None
for cur, dirs, files in os.walk("/kaggle/input"):
    if "train" in dirs and "test" in dirs:
        images_root = cur
        dirs[:] = [d for d in dirs if d not in ("train", "test")]
        break
assert images_root is not None, "Attach felipekitamura/unifesp-xray-bodypart-classification (Notebook Settings -> Data)."
print("Images dataset:", images_root)
IMAGES_DATASET = images_root

## Source files (generated from `experiments/v2/src/` -- do not edit here)

In [ ]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/__init__.py", "w", encoding="utf-8") as f:
    f.write('')
print("wrote src/__init__.py")

In [ ]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/config.py", "w", encoding="utf-8") as f:
    f.write('"""\nCentral config. Two roots, because on Kaggle they\'ll usually be two\ndifferent attached datasets:\n  - DATA_ROOT: where train/, test/ (the DICOM images) live\n  - INDEX_ROOT: where index.csv and splits/folds.csv live\n\nLocally both default to the repo root. On Kaggle, set them explicitly as\nenvironment variables in the notebook\'s first cell, e.g.:\n    import os\n    os.environ["UNIFESP_DATA_ROOT"] = "/kaggle/input/unifesp-xray-bodypart-classification"\n    os.environ["UNIFESP_INDEX_ROOT"] = "/kaggle/input/unifesp-index-files"  # your uploaded index.csv+splits/\n    os.environ["UNIFESP_OUTPUT_DIR"] = "/kaggle/working"\n(see KAGGLE_SETUP.md for the full first-cell snippet.)\n"""\nimport os\n\n_THIS_DIR = os.path.dirname(os.path.abspath(__file__))\n_REPO_ROOT = os.path.dirname(_THIS_DIR)\n\nON_KAGGLE = os.path.exists("/kaggle/input")\n\nif ON_KAGGLE:\n    DATA_ROOT = os.environ.get("UNIFESP_DATA_ROOT", "/kaggle/input")\n    INDEX_ROOT = os.environ.get("UNIFESP_INDEX_ROOT", DATA_ROOT)\n    OUTPUT_DIR = os.environ.get("UNIFESP_OUTPUT_DIR", "/kaggle/working")\nelse:\n    DATA_ROOT = os.environ.get("UNIFESP_DATA_ROOT", _REPO_ROOT)\n    INDEX_ROOT = os.environ.get("UNIFESP_INDEX_ROOT", DATA_ROOT)\n    OUTPUT_DIR = os.environ.get("UNIFESP_OUTPUT_DIR", os.path.join(_REPO_ROOT, "runs"))\n\nINDEX_CSV = os.path.join(INDEX_ROOT, "index.csv")\nFOLDS_CSV = os.path.join(INDEX_ROOT, "splits", "folds.csv")\n\n# Optional: where cache_images.py pre-decodes DICOM -> PNG once, so every\n# later training run reads a small PNG instead of re-decoding DICOM every\n# epoch (top Kaggle finishers reported this cuts training from hours to\n# minutes). None (default) means "no cache, decode DICOM every time" --\n# fully backward compatible, nothing breaks if you skip this step.\nCACHE_DIR = os.environ.get("UNIFESP_CACHE_DIR", os.path.join(OUTPUT_DIR, "image_cache") if ON_KAGGLE else None)\n\nif os.environ.get("UNIFESP_CACHE_DIR", "").lower() in ("none", "off", "0"):\n    CACHE_DIR = None        # no PNG cache (e.g. MURA / COVID: PNG decoding is cheap, a 40k-image cache would fill the disk)\n\n# Which dataset this run targets -- "unifesp" (default), "chestxray14",\n# "mura", or "covidcxr". Switching this is the ONLY config change needed to\n# retarget the whole pipeline (train.py/train_hybrid.py/evaluate_on_holdout.py/\n# etc. are all dataset-agnostic): it picks the label set below (-> last-layer\n# neuron count, via build_model\'s/HybridCNNTransformer\'s default n_classes=\n# C.N_CLASSES) and, in dataset.py, the image file format to load (.dcm vs\n# .png). Still set UNIFESP_DATA_ROOT to that dataset\'s own attached path.\nDATASET = os.environ.get("UNIFESP_DATASET", "unifesp")\n\nif DATASET == "chestxray14":\n    # NIH ChestX-ray14: 14 disease classes. "No Finding" is deliberately not\n    # its own class -- it\'s the all-zero label vector (same multilabel\n    # convention the CheXNet-style literature on this dataset uses), which\n    # engine.py\'s sample_mean_f1/compute_metrics already handle correctly.\n    LABEL_NAMES = {\n        0: "Atelectasis", 1: "Cardiomegaly", 2: "Consolidation", 3: "Edema",\n        4: "Effusion", 5: "Emphysema", 6: "Fibrosis", 7: "Hernia",\n        8: "Infiltration", 9: "Mass", 10: "Nodule", 11: "Pleural_Thickening",\n        12: "Pneumonia", 13: "Pneumothorax",\n    }\nelif DATASET == "mura":\n    # Stanford MURA, re-tasked as BODY-PART classification (the body part is\n    # already encoded in each image\'s folder name, XR_ELBOW/XR_HAND/...) --\n    # the same task UNIFESP solves, on an independent patient population and\n    # imaging source. Single-label (one body part per image) rather than\n    # multilabel, but a one-hot row is just a degenerate multilabel row, so\n    # nothing downstream needs to know the difference.\n    LABEL_NAMES = {\n        0: "Elbow", 1: "Finger", 2: "Forearm", 3: "Hand",\n        4: "Humerus", 5: "Shoulder", 6: "Wrist",\n    }\nelif DATASET == "covidcxr":\n    # COVID-19 Radiography Database (Qatar University + University of Dhaka),\n    # re-tasked as 4-class chest X-ray diagnosis classification. Single-label\n    # (one diagnosis per image), aggregated from many source datasets/\n    # publications with no consistent patient-id field across them, so\n    # build_index_covidcxr.py uses each image\'s own uid as its "patient_id"\n    # -- an image-level split, not patient-level (a limitation worth naming\n    # in the paper, same as why the other datasets\' patient-level splitting\n    # matters in the first place).\n    LABEL_NAMES = {\n        0: "COVID", 1: "Lung_Opacity", 2: "Normal", 3: "Viral_Pneumonia",\n    }\nelse:\n    LABEL_NAMES = {\n        0: "Abdomen", 1: "Ankle", 2: "Cervical Spine", 3: "Chest", 4: "Clavicles",\n        5: "Elbow", 6: "Feet", 7: "Finger", 8: "Forearm", 9: "Hand", 10: "Hip",\n        11: "Knee", 12: "Lower Leg", 13: "Lumbar Spine", 14: "Others", 15: "Pelvis",\n        16: "Shoulder", 17: "Sinus", 18: "Skull", 19: "Thigh", 20: "Thoracic Spine",\n        21: "Wrist",\n    }\nN_CLASSES = len(LABEL_NAMES)\nIMG_SIZE = 256\nSEED = 42\n\nos.makedirs(OUTPUT_DIR, exist_ok=True)\n')
print("wrote src/config.py")

In [ ]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/dataset.py", "w", encoding="utf-8") as f:
    f.write('"""\nUNIFESP DICOM dataset for multilabel body-part classification.\n\nHandles the two DICOM quirks we found in EDA:\n  - MONOCHROME1 vs MONOCHROME2 photometric interpretation (2367 vs 114 files)\n    -> MONOCHROME1 is inverted, so we flip it to a consistent polarity.\n  - Variable BitsStored (8/10/12/15-bit) -> min-max normalize per image instead\n    of assuming a fixed bit depth.\n"""\nimport os\nimport numpy as np\nimport pydicom\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset\n\nfrom . import config as C\n\n\ndef load_cached_or_decode(uid, src_path, cache_dir):\n    """Read uid\'s pre-cached PNG if cache_dir has one; otherwise decode the\n    source image (DICOM for unifesp, PNG passthrough for chestxray14 -- see\n    load_image_for_dataset) and, if cache_dir is set, write the cache PNG\n    for next time. Write is atomic (temp file + os.replace) so two training\n    processes caching the same image concurrently (e.g. two backbones on two\n    GPUs both hitting fold 0\'s validation set) never leaves a half-written\n    file for the other to read -- worst case both just redundantly decode it\n    once.\n    """\n    if cache_dir is not None:\n        png_path = os.path.join(cache_dir, f"{uid}.png")\n        if os.path.exists(png_path):\n            return np.array(Image.open(png_path).convert("RGB"))\n\n    image = load_image_for_dataset(src_path)\n\n    if cache_dir is not None:\n        os.makedirs(cache_dir, exist_ok=True)\n        tmp_path = os.path.join(cache_dir, f".{uid}.{os.getpid()}.tmp.png")\n        Image.fromarray(image).save(tmp_path)\n        os.replace(tmp_path, os.path.join(cache_dir, f"{uid}.png"))\n\n    return image\n\n\ndef load_image_for_dataset(path):\n    """Dispatch on C.DATASET: DICOM for unifesp, plain PNG for chestxray14/mura/covidcxr."""\n    if C.DATASET in ("chestxray14", "mura", "covidcxr"):\n        return load_png_as_image(path)\n    return load_dicom_as_image(path)\n\n\ndef load_png_as_image(path):\n    arr = np.array(Image.open(path).convert("L"))  # ChestX-ray14 PNGs are grayscale (some stored as RGBA)\n    arr = np.stack([arr, arr, arr], axis=-1)  # 1-channel X-ray -> pseudo-RGB for ImageNet backbones\n    return arr.astype(np.uint8)\n\n\ndef load_dicom_as_image(path):\n    ds = pydicom.dcmread(path, force=True)\n    arr = ds.pixel_array.astype(np.float32)\n\n    slope = float(getattr(ds, "RescaleSlope", 1.0))\n    intercept = float(getattr(ds, "RescaleIntercept", 0.0))\n    arr = arr * slope + intercept\n\n    if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":\n        arr = arr.max() - arr\n\n    lo, hi = arr.min(), arr.max()\n    arr = (arr - lo) / (hi - lo + 1e-6)\n    arr = np.stack([arr, arr, arr], axis=-1)  # 1-channel X-ray -> pseudo-RGB for ImageNet backbones\n    return (arr * 255.0).astype(np.uint8)\n\n\ndef _basename_uid(path):\n    b = os.path.basename(path)[:-4]  # strip .dcm\n    if b.endswith("-c"):\n        b = b[:-2]\n    return b\n\n\ndef build_uid_to_path(data_root):\n    """Walk data_root once and map uid -> absolute file path. We do this at\n    runtime instead of trusting a stored relative path, because the raw\n    image folders are nested inconsistently (a local zip-extract quirk gave\n    us train/train/train/<id>/..., while the Kaggle-mounted dataset or a\n    re-extraction elsewhere may only have train/<id>/...). Matching by the\n    filename\'s UID instead of a fixed path makes this robust to wherever the\n    raw files actually end up on disk."""\n    if C.DATASET == "chestxray14":\n        lookup = {}\n        for root, _dirs, files in os.walk(data_root):\n            for f in files:\n                # skip macOS AppleDouble metadata files (e.g. "._image1.png") --\n                # they match the .png extension but aren\'t real images, and\n                # crash PIL with UnidentifiedImageError (seen on the Kaggle\n                # mirror of this dataset, which was zipped from a Mac)\n                if f.lower().endswith(".png") and not f.startswith("._"):\n                    lookup[f[:-4]] = os.path.join(root, f)\n        return lookup\n\n    if C.DATASET in ("mura", "covidcxr"):\n        # MURA\'s own filenames (image1.png, image2.png, ...) repeat in every\n        # study folder, so the bare filename can\'t be a uid -- use the full\n        # path relative to data_root instead (unique, and keeps the\n        # body-part/class-folder info parseable by build_index_mura.py /\n        # build_index_covidcxr.py). covidcxr also has a parallel "masks/"\n        # tree of lung-segmentation masks with the same filenames as\n        # "images/" -- skip it here so masks never get treated as data.\n        lookup = {}\n        for root, _dirs, files in os.walk(data_root):\n            if C.DATASET == "covidcxr" and os.path.basename(root).lower() == "masks":\n                continue\n            for f in files:\n                # skip macOS AppleDouble metadata files (e.g. "._image1.png") --\n                # they match the .png extension but aren\'t real images, and\n                # crash PIL with UnidentifiedImageError (seen on the Kaggle\n                # mirror of MURA, which was zipped from a Mac)\n                if f.lower().endswith(".png") and not f.startswith("._"):\n                    full_path = os.path.join(root, f)\n                    rel = os.path.relpath(full_path, data_root)\n                    uid = rel.replace(os.sep, "__")[:-4]\n                    lookup[uid] = full_path\n        return lookup\n\n    lookup = {}\n    for root, _dirs, files in os.walk(data_root):\n        for f in files:\n            if f.lower().endswith(".dcm") and not f.startswith("._"):\n                lookup[_basename_uid(f)] = os.path.join(root, f)\n    return lookup\n\n\nclass UnifespDataset(Dataset):\n    def __init__(self, df, data_root, transform=None, has_labels=True, uid_to_path=None, cache_dir=C.CACHE_DIR):\n        """\n        df: a slice of index.csv (optionally merged with folds.csv), must have a \'uid\' column\n        data_root: base dir containing the (possibly nested) train/test DICOM folders\n        uid_to_path: pass a precomputed dict to avoid re-walking data_root for every split\n        cache_dir: optional PNG cache (see load_cached_or_decode); None disables caching\n        """\n        self.df = df.reset_index(drop=True)\n        self.data_root = data_root\n        self.transform = transform\n        self.has_labels = has_labels\n        self.label_cols = [f"label_{k}_{C.LABEL_NAMES[k].replace(\' \', \'_\')}" for k in range(C.N_CLASSES)]\n        self.uid_to_path = uid_to_path if uid_to_path is not None else build_uid_to_path(data_root)\n        self.cache_dir = cache_dir\n\n        missing = set(self.df["uid"]) - set(self.uid_to_path)\n        if missing:\n            raise FileNotFoundError(\n                f"{len(missing)} uid(s) in this split have no matching .dcm file under {data_root} "\n                f"(e.g. {next(iter(missing))}). Check UNIFESP_DATA_ROOT points at the right dataset."\n            )\n\n    def __len__(self):\n        return len(self.df)\n\n    def __getitem__(self, idx):\n        row = self.df.iloc[idx]\n        uid = row["uid"]\n        path = self.uid_to_path[uid]\n        image = load_cached_or_decode(uid, path, self.cache_dir)\n\n        if self.transform is not None:\n            image = self.transform(image=image)["image"]\n\n        if self.has_labels:\n            target = row[self.label_cols].to_numpy(dtype=np.float32)\n            target = torch.from_numpy(target)\n            return image, target\n        return image, row["uid"]\n\n\ndef get_transforms(img_size=C.IMG_SIZE, train=True):\n    """Train-time augmentation.\n\n    We tried matching the UNIFESP Kaggle competition\'s top finishers\n    (2nd place writeup used GaussNoise + GaussianBlur + ShiftScaleRotate +\n    brightness/contrast) and measured it head-to-head on our own fold 0:\n\n        backbone          light-aug  heavy-aug(+noise/blur)\n        efficientnetv2_s     0.876      0.851   (worse)\n        efficientnet_b3      0.866      0.847   (worse)\n        inception_v3         0.872      0.841   (worse)\n        convnext_tiny        0.738      0.558   (much worse)\n        densenet121          0.869      0.873   (~same)\n        vgg16                0.827      0.836   (~same)\n\n    GaussNoise/GaussianBlur made 6/8 backbones worse (macro_f1), ConvNeXt-\n    Tiny catastrophically so -- likely because blurring removes the fine\n    bone-edge texture that separates visually similar classes (Lumbar vs\n    Thoracic Spine, Hand vs Wrist), and our fixed 30-epoch budget (no\n    warmup, no early stopping) doesn\'t give heavier augmentation enough\n    iterations to pay off the way it did for the competition\'s longer,\n    early-stopped fastai runs. So: geometric augmentation (flip, mild\n    shift/scale/rotate) + brightness/contrast, no synthetic noise/blur.\n    This negative result is itself worth keeping for the paper\'s ablation\n    section.\n    """\n    import albumentations as A\n    from albumentations.pytorch import ToTensorV2\n\n    mean = (0.485, 0.456, 0.406)\n    std = (0.229, 0.224, 0.225)\n\n    if train:\n        return A.Compose([\n            A.Resize(img_size, img_size),\n            A.HorizontalFlip(p=0.5),\n            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.5),\n            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),\n            A.Normalize(mean=mean, std=std),\n            ToTensorV2(),\n        ])\n    return A.Compose([\n        A.Resize(img_size, img_size),\n        A.Normalize(mean=mean, std=std),\n        ToTensorV2(),\n    ])\n')
print("wrote src/dataset.py")

In [ ]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/build_index.py", "w", encoding="utf-8") as f:
    f.write('"""\nStep 1a: Build a flat index of the UNIFESP dataset.\n\nReads every DICOM\'s header (no pixel decode, so it\'s fast) and writes a\nsingle flat CSV that the rest of the pipeline uses. Uses config.DATA_ROOT /\nconfig.INDEX_ROOT (not a path relative to this file) so it works no matter\nwhere the raw images or the output location actually are -- e.g. run fresh\ninside a Kaggle notebook against the attached images dataset, writing to\n/kaggle/working, with no separately-uploaded "index" dataset needed at all.\n\nOutput: <INDEX_ROOT>/index.csv with columns:\n    uid, split, patient_id, study_uid, series_uid, labels, label_0_Abdomen, ...\n(no filepath column -- dataset.py locates files by uid via build_uid_to_path,\nwhich is robust to whatever nesting the raw folders happen to have.)\n"""\nimport os\nimport pydicom\nimport pandas as pd\n\nfrom . import config as C\nfrom .dataset import build_uid_to_path\n\nLABEL_NAMES = C.LABEL_NAMES\nN_CLASSES = C.N_CLASSES\n\n\ndef main():\n    print("Indexing DICOM files under", C.DATA_ROOT, "...")\n    train_uid_to_path = build_uid_to_path(os.path.join(C.DATA_ROOT, "train"))\n    test_uid_to_path = build_uid_to_path(os.path.join(C.DATA_ROOT, "test"))\n    print(f"train dcm files: {len(train_uid_to_path)} | test dcm files: {len(test_uid_to_path)}")\n\n    rows = []\n    for split, uid_to_path in (("train", train_uid_to_path), ("test", test_uid_to_path)):\n        for uid, path in uid_to_path.items():\n            try:\n                ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)\n                patient_id = str(getattr(ds, "PatientID", ""))\n                study_uid = str(getattr(ds, "StudyInstanceUID", ""))\n                series_uid = str(getattr(ds, "SeriesInstanceUID", ""))\n            except Exception as e:\n                patient_id = study_uid = series_uid = ""\n                print("WARN: failed to read header for", path, e)\n            rows.append({\n                "uid": uid, "split": split,\n                "patient_id": patient_id, "study_uid": study_uid, "series_uid": series_uid,\n            })\n\n    index_df = pd.DataFrame(rows)\n\n    # attach multilabel targets for the train split\n    train_csv = pd.read_csv(os.path.join(C.DATA_ROOT, "train.csv"))\n    train_csv["SOPInstanceUID"] = train_csv["SOPInstanceUID"].str.strip()\n    label_map = dict(zip(train_csv["SOPInstanceUID"], train_csv["Target"]))\n    index_df["labels"] = index_df["uid"].map(label_map)\n\n    # one-hot columns for the 22 classes (NaN/absent for test rows)\n    for k in range(N_CLASSES):\n        col = f"label_{k}_{LABEL_NAMES[k].replace(\' \', \'_\')}"\n        index_df[col] = index_df["labels"].apply(\n            lambda s, k=k: int(k in [int(x) for x in str(s).split()]) if pd.notna(s) else pd.NA\n        )\n\n    os.makedirs(C.INDEX_ROOT, exist_ok=True)\n    out_path = os.path.join(C.INDEX_ROOT, "index.csv")\n    index_df.to_csv(out_path, index=False)\n    print("Saved index to", out_path, "| rows:", len(index_df))\n\n    # ---- patient-level leakage sanity check ----\n    train_idx = index_df[index_df["split"] == "train"]\n    test_idx = index_df[index_df["split"] == "test"]\n    train_patients = set(train_idx["patient_id"]) - {""}\n    test_patients = set(test_idx["patient_id"]) - {""}\n    overlap = train_patients & test_patients\n    print(f"\\nUnique patients -> train: {len(train_patients)}, test: {len(test_patients)}")\n    print(f"Patient overlap between train/test (Kaggle\'s own split): {len(overlap)}")\n\n    dup = train_idx.groupby("patient_id").size()\n    multi_image_patients = (dup > 1).sum()\n    print(f"Train patients with >1 image: {multi_image_patients} / {len(dup)}")\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote src/build_index.py")

In [ ]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/make_splits.py", "w", encoding="utf-8") as f:
    f.write('"""\nStep 1b: Patient-level, multilabel-stratified K-fold split.\n\nWhy patient-level: build_index.py found 483/768 train patients have >1 image,\nso a plain random (or even plain multilabel-stratified) image-level split can\nput the same patient\'s images in both train and validation -> leaky, overly\noptimistic validation scores. We group by patient_id first, then stratify.\n\nWhy multilabel-stratified: UNIFESP\'s 22 classes are heavily imbalanced\n(Chest 738 vs Clavicles 9), so a plain GroupKFold can starve a fold of a rare\nclass entirely. We stratify on each patient\'s aggregated (union) label\nvector using iterative stratification.\n\nOutput: splits/fold_{i}.csv with columns [uid, fold] for every train image.\n"""\nimport os\nimport numpy as np\nimport pandas as pd\nfrom iterstrat.ml_stratifiers import MultilabelStratifiedKFold\n\nfrom . import config as C\n\nN_CLASSES = C.N_CLASSES\nN_FOLDS = 5\nSEED = C.SEED\n\n\ndef main():\n    index_df = pd.read_csv(os.path.join(C.INDEX_ROOT, "index.csv"))\n    train_df = index_df[index_df["split"] == "train"].copy()\n\n    label_cols = [c for c in train_df.columns if c.startswith("label_")]\n    assert len(label_cols) == N_CLASSES, label_cols\n\n    # aggregate to patient level: union of all labels this patient\'s images carry\n    patient_labels = (\n        train_df.groupby("patient_id")[label_cols]\n        .max()\n        .reset_index()\n    )\n    patients = patient_labels["patient_id"].values\n    Y = patient_labels[label_cols].values.astype(int)\n\n    print(f"Patients: {len(patients)} | images: {len(train_df)}")\n    print("Per-class positive patient count:\\n", pd.Series(Y.sum(axis=0), index=label_cols))\n\n    mskf = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)\n    patient_fold = np.full(len(patients), -1, dtype=int)\n    for fold, (_, val_idx) in enumerate(mskf.split(patients, Y)):\n        patient_fold[val_idx] = fold\n\n    patient_to_fold = dict(zip(patients, patient_fold))\n    train_df["fold"] = train_df["patient_id"].map(patient_to_fold)\n\n    assert (train_df["fold"] >= 0).all(), "every image must get a fold"\n\n    out_cols = ["uid", "patient_id", "fold"] + label_cols\n    splits_dir = os.path.join(C.INDEX_ROOT, "splits")\n    os.makedirs(splits_dir, exist_ok=True)\n    out_path = os.path.join(splits_dir, "folds.csv")\n    train_df[out_cols].to_csv(out_path, index=False)\n    print("Saved", out_path)\n\n    # sanity: no patient split across folds, per-fold class coverage\n    leak = train_df.groupby("patient_id")["fold"].nunique()\n    print("Patients appearing in >1 fold (should be 0):", (leak > 1).sum())\n\n    print("\\nPer-fold image count:\\n", train_df["fold"].value_counts().sort_index())\n    print("\\nPer-fold positive image count per class:")\n    print(train_df.groupby("fold")[label_cols].sum())\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote src/make_splits.py")

In [ ]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/make_holdout_cv_splits.py", "w", encoding="utf-8") as f:
    f.write('"""\nNotebook 2\'s split scheme: a permanent, never-trained-on held-out test set,\nplus a fresh 5-fold CV split restricted to the remaining pool.\n\nThis is different from make_splits.py\'s plain 5-fold split (where every\nfold is, in turn, somebody\'s validation set -- fine for model *selection*,\nbut every image still ends up influencing model checkpointing at some\npoint). Here, one of the original 5 folds (--holdout_fold, default 4) is\ncarved out as a fixed test set that no training run -- across any of the\n5 new CV folds -- ever sees, train or val. Each CV fold\'s best checkpoint\nis later evaluated against this same fixed set (see evaluate_on_holdout.py),\ngiving a genuine mean +/- std generalization estimate instead of 5 folds\nthat are each other\'s test set.\n\nPatient-level throughout (same reasoning as make_splits.py: many UNIFESP\npatients have >1 image, so splitting by image would leak).\n\nOutput:\n    <INDEX_ROOT>/splits/holdout_uids.csv   -- uid column, the fixed test set\n    <INDEX_ROOT>/splits/cv_folds.csv       -- uid, cv_fold (0..4), pool only\n\nUsage (after make_splits.py has already produced splits/folds.csv):\n    python -m src.make_holdout_cv_splits --holdout_fold 4\n"""\nimport argparse\nimport os\n\nimport numpy as np\nimport pandas as pd\nfrom iterstrat.ml_stratifiers import MultilabelStratifiedKFold\n\nfrom . import config as C\n\nN_CLASSES = C.N_CLASSES\nN_CV_FOLDS = 5\nSEED = C.SEED\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--holdout_fold", type=int, default=4,\n                    help="which of the original 5 folds (from make_splits.py) becomes the fixed holdout test set")\n    args = p.parse_args()\n\n    folds_df = pd.read_csv(C.FOLDS_CSV)\n    label_cols = [c for c in folds_df.columns if c.startswith("label_")]\n    assert len(label_cols) == N_CLASSES, label_cols\n\n    holdout_df = folds_df[folds_df["fold"] == args.holdout_fold].reset_index(drop=True)\n    pool_df = folds_df[folds_df["fold"] != args.holdout_fold].reset_index(drop=True)\n    print(f"Holdout test (original fold {args.holdout_fold}): {len(holdout_df)} images, "\n          f"{holdout_df[\'patient_id\'].nunique()} patients")\n    print(f"CV pool (remaining images): {len(pool_df)} images, {pool_df[\'patient_id\'].nunique()} patients "\n          f"({len(pool_df) / len(folds_df):.1%} of all train images)")\n\n    # fresh patient-level multilabel-stratified 5-fold split, restricted to the pool only\n    patient_labels = pool_df.groupby("patient_id")[label_cols].max().reset_index()\n    patients = patient_labels["patient_id"].values\n    Y = patient_labels[label_cols].values.astype(int)\n\n    mskf = MultilabelStratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=SEED)\n    patient_cv_fold = np.full(len(patients), -1, dtype=int)\n    for cv_fold, (_, val_idx) in enumerate(mskf.split(patients, Y)):\n        patient_cv_fold[val_idx] = cv_fold\n    patient_to_cv_fold = dict(zip(patients, patient_cv_fold))\n    pool_df = pool_df.copy()\n    pool_df["cv_fold"] = pool_df["patient_id"].map(patient_to_cv_fold)\n    assert (pool_df["cv_fold"] >= 0).all()\n\n    # sanity: no patient split across folds or across the holdout boundary\n    leak = pool_df.groupby("patient_id")["cv_fold"].nunique()\n    assert (leak <= 1).all(), "a pool patient landed in >1 cv_fold"\n    assert set(holdout_df["patient_id"]) & set(pool_df["patient_id"]) == set(), \\\n        "a patient appears in both the holdout set and the CV pool"\n\n    print("\\nPer-cv_fold image count:\\n", pool_df["cv_fold"].value_counts().sort_index())\n    print("\\nPer-cv_fold positive image count per class:")\n    print(pool_df.groupby("cv_fold")[label_cols].sum())\n    print("\\nHoldout set positive image count per class:")\n    print(holdout_df[label_cols].sum())\n\n    splits_dir = os.path.join(C.INDEX_ROOT, "splits")\n    os.makedirs(splits_dir, exist_ok=True)\n\n    holdout_out = os.path.join(splits_dir, "holdout_uids.csv")\n    holdout_df[["uid", "patient_id"] + label_cols].to_csv(holdout_out, index=False)\n    print("\\nSaved:", holdout_out)\n\n    cv_out = os.path.join(splits_dir, "cv_folds.csv")\n    pool_df[["uid", "patient_id", "cv_fold"] + label_cols].rename(columns={"cv_fold": "fold"}).to_csv(\n        cv_out, index=False\n    )\n    print("Saved:", cv_out)\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote src/make_holdout_cv_splits.py")

In [ ]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/cache_images.py", "w", encoding="utf-8") as f:
    f.write('"""\nOptional one-time speed-up: pre-decode every DICOM into a PNG cache before\nlaunching the 8 parallel training runs. UnifespDataset already caches\nlazily on first read (see dataset.py\'s load_cached_or_decode), so this\nscript isn\'t required for correctness -- it just avoids every one of the\nfirst 2 (parallel) backbones redundantly decoding the same ~1700 training\nimages during their first epoch. Top Kaggle finishers on this exact\ncompetition reported DICOM decoding as their biggest bottleneck.\n\nKaggle usage, right after environment setup, before Step 2 training:\n    python -m src.cache_images\n"""\nimport time\n\nfrom . import config as C\nfrom .dataset import build_uid_to_path, load_cached_or_decode\n\n\ndef main():\n    if C.CACHE_DIR is None:\n        print("UNIFESP_CACHE_DIR / CACHE_DIR is not set -- nothing to do. "\n              "Caching is off by default outside Kaggle; set the env var if you want it locally.")\n        return\n\n    ext = "PNG" if C.DATASET in ("chestxray14", "mura", "covidcxr") else "DICOM"\n    print(f"Indexing {ext} files under", C.DATA_ROOT, "...")\n    uid_to_path = build_uid_to_path(C.DATA_ROOT)\n    print(f"Found {len(uid_to_path)} files. Caching to {C.CACHE_DIR} ...")\n\n    t0 = time.time()\n    for i, (uid, path) in enumerate(uid_to_path.items()):\n        load_cached_or_decode(uid, path, C.CACHE_DIR)\n        if (i + 1) % 250 == 0:\n            print(f"  {i + 1}/{len(uid_to_path)} ({time.time() - t0:.1f}s elapsed)")\n\n    print(f"Done. Cached {len(uid_to_path)} images in {time.time() - t0:.1f}s -> {C.CACHE_DIR}")\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote src/cache_images.py")

In [ ]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/evalkit.py", "w", encoding="utf-8") as f:
    f.write('"""\nevalkit -- leakage-free evaluation utilities for the UNIFESP body-part project.\n\nPure numpy (no torch), so it runs anywhere: Kaggle, laptop, CI.\n\nWhat it fixes compared with the earlier notebooks\n-------------------------------------------------\n1. Thresholds are tuned on OUT-OF-FOLD (OOF) predictions of the CV pool and then\n   FROZEN before they touch the held-out set (``tune_thresholds_oof`` +\n   ``nested_evaluate``).  The old ``ensemble_eval.py`` tuned thresholds on the very\n   fold it then scored, which inflated sample_mean_f1 (0.9356 / 0.9445).\n2. "At least one label per image" decoding (``predict_labels(min_one=True)``): ~92 %\n   of UNIFESP images are single-label and an empty prediction scores F1 = 0.\n3. Patient-level (cluster) bootstrap confidence intervals and a paired bootstrap\n   test for model-vs-model differences (``cluster_bootstrap``, ``paired_bootstrap``).\n4. Caruana greedy ensemble selection on OOF predictions (``caruana_ensemble``).\n5. Kaggle submission writer + validator (``write_submission``, ``check_submission``).\n\nMetric\n------\n``sample_mean_f1`` = per-image F1 between predicted and true label sets, averaged over\nimages.  This is the Kaggle "Mean F1-Score" of the UNIFESP competition.\n"""\nfrom __future__ import annotations\n\nimport csv\nfrom typing import Iterable, Sequence\n\nimport numpy as np\n\nN_CLASSES = 22\nDEFAULT_GRID = np.round(np.arange(0.05, 0.951, 0.05), 2)\n\n\n# --------------------------------------------------------------------------- metrics\ndef per_image_f1(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:\n    """F1 per image. Both-empty -> 1.0, true-nonempty & pred-empty -> 0.0."""\n    y_true = np.asarray(y_true).astype(np.int64)\n    y_pred = np.asarray(y_pred).astype(np.int64)\n    tp = (y_true & y_pred).sum(1)\n    fp = ((1 - y_true) & y_pred).sum(1)\n    fn = (y_true & (1 - y_pred)).sum(1)\n    denom = 2 * tp + fp + fn\n    return np.where(denom == 0, 1.0, 2.0 * tp / np.maximum(denom, 1))\n\n\ndef sample_mean_f1(y_true, y_pred) -> float:\n    return float(per_image_f1(y_true, y_pred).mean())\n\n\ndef macro_f1(y_true, y_pred) -> float:\n    """Mean of per-class F1 (zero_division=0, same as sklearn\'s default for this use)."""\n    y_true = np.asarray(y_true).astype(np.int64)\n    y_pred = np.asarray(y_pred).astype(np.int64)\n    tp = (y_true & y_pred).sum(0)\n    fp = ((1 - y_true) & y_pred).sum(0)\n    fn = (y_true & (1 - y_pred)).sum(0)\n    denom = 2 * tp + fp + fn\n    f1 = np.where(denom == 0, 0.0, 2.0 * tp / np.maximum(denom, 1))\n    return float(f1.mean())\n\n\ndef micro_f1(y_true, y_pred) -> float:\n    y_true = np.asarray(y_true).astype(np.int64)\n    y_pred = np.asarray(y_pred).astype(np.int64)\n    tp = (y_true & y_pred).sum()\n    fp = ((1 - y_true) & y_pred).sum()\n    fn = (y_true & (1 - y_pred)).sum()\n    denom = 2 * tp + fp + fn\n    return float(0.0 if denom == 0 else 2.0 * tp / denom)\n\n\ndef subset_accuracy(y_true, y_pred) -> float:\n    return float((np.asarray(y_true) == np.asarray(y_pred)).all(axis=1).mean())\n\n\ndef all_metrics(y_true, y_pred) -> dict:\n    return {\n        "sample_mean_f1": sample_mean_f1(y_true, y_pred),\n        "macro_f1": macro_f1(y_true, y_pred),\n        "micro_f1": micro_f1(y_true, y_pred),\n        "subset_accuracy": subset_accuracy(y_true, y_pred),\n    }\n\n\n# --------------------------------------------------------------------------- decoding\ndef predict_labels(prob: np.ndarray, thresholds=0.5, min_one: bool = True,\n                   max_labels: int | None = None) -> np.ndarray:\n    """Probabilities -> 0/1 matrix.\n\n    min_one:    every image gets at least its best-scoring class (score = prob / threshold,\n                so a class with a low tuned threshold is not unfairly skipped).\n    max_labels: keep at most this many labels per image (by the same score).\n    """\n    prob = np.asarray(prob, dtype=np.float64)\n    thr = np.broadcast_to(np.asarray(thresholds, dtype=np.float64), (prob.shape[1],))\n    pred = (prob >= thr[None, :]).astype(np.int64)\n    score = prob / np.maximum(thr[None, :], 1e-6)\n    if min_one:\n        empty = pred.sum(1) == 0\n        if empty.any():\n            best = score[empty].argmax(1)\n            pred[np.where(empty)[0], best] = 1\n    if max_labels is not None:\n        order = np.argsort(-score, axis=1)\n        keep = np.zeros_like(pred)\n        rows = np.arange(prob.shape[0])[:, None]\n        keep[rows, order[:, :max_labels]] = 1\n        pred = pred * keep\n        if min_one:  # top-1 is always kept, but be explicit\n            pred[np.arange(prob.shape[0]), order[:, 0]] = 1\n    return pred\n\n\n# --------------------------------------------------------------------------- thresholds\ndef tune_thresholds_oof(oof_prob: np.ndarray, y: np.ndarray, grid: Sequence[float] = DEFAULT_GRID,\n                        base: float = 0.5, n_rounds: int = 3, min_pos: float = 20.0,\n                        min_one: bool = True) -> np.ndarray:\n    """Per-class thresholds by coordinate ascent on the REAL objective (sample_mean_f1).\n\n    * Use ONLY out-of-fold predictions of the CV pool here -- never the held-out set and\n      never the same fold you later report on.\n    * Rare classes are shrunk toward ``base`` with weight n_pos / (n_pos + min_pos), because\n      a threshold fitted on 5-10 positives is mostly noise.\n    """\n    oof_prob = np.asarray(oof_prob, dtype=np.float64)\n    y = np.asarray(y).astype(np.int64)\n    C = oof_prob.shape[1]\n    thr = np.full(C, base, dtype=np.float64)\n\n    def score(t):\n        return sample_mean_f1(y, predict_labels(oof_prob, t, min_one=min_one))\n\n    best = score(thr)\n    for _ in range(n_rounds):\n        changed = False\n        for c in range(C):\n            cur = thr[c]\n            best_t, best_s = cur, best\n            for g in grid:\n                if g == cur:\n                    continue\n                thr[c] = g\n                s = score(thr)\n                # strictly better, or equal but closer to the base threshold (prefer simple)\n                if s > best_s + 1e-12 or (abs(s - best_s) <= 1e-12 and abs(g - base) < abs(best_t - base)):\n                    best_t, best_s = g, s\n            thr[c] = best_t\n            if best_t != cur:\n                changed = True\n            best = best_s\n        if not changed:\n            break\n    n_pos = y.sum(0).astype(np.float64)\n    lam = n_pos / (n_pos + min_pos)\n    return lam * thr + (1.0 - lam) * base\n\n\ndef nested_evaluate(oof_prob, y_oof, hold_prob, y_hold, **tune_kw) -> dict:\n    """The only honest way to report tuned-threshold numbers.\n\n    thresholds <- OOF of the CV pool;  scores <- held-out set (never used for any choice).\n    Returns the metrics for three decoding rules so each lever\'s gain is visible.\n    """\n    thr = tune_thresholds_oof(oof_prob, y_oof, **tune_kw)\n    out = {\n        "thresholds": thr,\n        "hold_0.5": all_metrics(y_hold, predict_labels(hold_prob, 0.5, min_one=False)),\n        "hold_0.5_min_one": all_metrics(y_hold, predict_labels(hold_prob, 0.5, min_one=True)),\n        "hold_tuned_min_one": all_metrics(y_hold, predict_labels(hold_prob, thr, min_one=True)),\n    }\n    return out\n\n\n# --------------------------------------------------------------------------- bootstrap\ndef _cluster_indices(groups: np.ndarray):\n    groups = np.asarray(groups)\n    uniq, inv = np.unique(groups, return_inverse=True)\n    members = [np.where(inv == k)[0] for k in range(len(uniq))]\n    return uniq, members\n\n\ndef cluster_bootstrap(metric_fn, y_true, y_pred, groups, n_boot: int = 2000, seed: int = 0,\n                      alpha: float = 0.05) -> dict:\n    """Patient-level bootstrap: resample PATIENTS (all their images together), recompute metric.\n\n    metric_fn(y_true, y_pred) -> float.  ``groups`` = patient id per image.\n    """\n    y_true = np.asarray(y_true)\n    y_pred = np.asarray(y_pred)\n    _, members = _cluster_indices(groups)\n    P = len(members)\n    rng = np.random.default_rng(seed)\n    vals = np.empty(n_boot)\n    for b in range(n_boot):\n        pick = rng.integers(0, P, size=P)\n        idx = np.concatenate([members[k] for k in pick])\n        vals[b] = metric_fn(y_true[idx], y_pred[idx])\n    lo, hi = np.quantile(vals, [alpha / 2, 1 - alpha / 2])\n    return {"estimate": float(metric_fn(y_true, y_pred)), "ci_low": float(lo), "ci_high": float(hi),\n            "boot_std": float(vals.std(ddof=1)), "n_patients": P, "n_boot": n_boot}\n\n\ndef paired_bootstrap(metric_fn, y_true, pred_a, pred_b, groups, n_boot: int = 2000, seed: int = 0,\n                     alpha: float = 0.05) -> dict:\n    """Is model A better than B?  Same patient resamples for both; returns CI of (A - B)."""\n    y_true = np.asarray(y_true)\n    pred_a = np.asarray(pred_a)\n    pred_b = np.asarray(pred_b)\n    _, members = _cluster_indices(groups)\n    P = len(members)\n    rng = np.random.default_rng(seed)\n    d = np.empty(n_boot)\n    for b in range(n_boot):\n        pick = rng.integers(0, P, size=P)\n        idx = np.concatenate([members[k] for k in pick])\n        d[b] = metric_fn(y_true[idx], pred_a[idx]) - metric_fn(y_true[idx], pred_b[idx])\n    lo, hi = np.quantile(d, [alpha / 2, 1 - alpha / 2])\n    # two-sided bootstrap p-value for H0: difference == 0\n    p = 2.0 * min((d <= 0).mean(), (d >= 0).mean())\n    return {"diff": float(metric_fn(y_true, pred_a) - metric_fn(y_true, pred_b)),\n            "ci_low": float(lo), "ci_high": float(hi), "p_value": float(min(1.0, p)), "n_boot": n_boot}\n\n\n# --------------------------------------------------------------------------- ensembling\ndef average_probs(prob_list: Iterable[np.ndarray], weights=None, mode: str = "prob") -> np.ndarray:\n    arr = np.stack([np.asarray(p, dtype=np.float64) for p in prob_list])\n    w = np.ones(len(arr)) if weights is None else np.asarray(weights, dtype=np.float64)\n    w = w / w.sum()\n    if mode == "logit":\n        eps = 1e-6\n        lg = np.log(np.clip(arr, eps, 1 - eps) / (1 - np.clip(arr, eps, 1 - eps)))\n        z = np.tensordot(w, lg, axes=1)\n        return 1.0 / (1.0 + np.exp(-z))\n    return np.tensordot(w, arr, axes=1)\n\n\ndef caruana_ensemble(oof_list: Sequence[np.ndarray], y: np.ndarray, n_iter: int = 40,\n                     init_best: bool = True) -> np.ndarray:\n    """Greedy forward selection WITH replacement on OOF sample_mean_f1 (0.5 thr, min_one).\n\n    Returns weights (sum 1).  Fit on OOF only; apply the weights unchanged to held-out/test.\n    """\n    k = len(oof_list)\n    counts = np.zeros(k)\n\n    def sc(w):\n        p = average_probs(oof_list, w)\n        return sample_mean_f1(y, predict_labels(p, 0.5, min_one=True))\n\n    if init_best:\n        singles = [sc(np.eye(k)[i]) for i in range(k)]\n        counts[int(np.argmax(singles))] = 1\n    for _ in range(n_iter):\n        best_i, best_s = None, -1.0\n        for i in range(k):\n            trial = counts.copy()\n            trial[i] += 1\n            s = sc(trial)\n            if s > best_s:\n                best_i, best_s = i, s\n        counts[best_i] += 1\n    return counts / counts.sum()\n\n\ndef combine_heads(p_sigmoid: np.ndarray, p_softmax: np.ndarray, alpha: float = 0.5) -> np.ndarray:\n    """Blend a multilabel (sigmoid) head with a single-label (softmax) head."""\n    return alpha * np.asarray(p_sigmoid) + (1.0 - alpha) * np.asarray(p_softmax)\n\n\n# --------------------------------------------------------------------------- submission\ndef write_submission(uids: Sequence[str], prob: np.ndarray, thresholds, path: str,\n                     min_one: bool = True, max_labels: int | None = None) -> np.ndarray:\n    pred = predict_labels(prob, thresholds, min_one=min_one, max_labels=max_labels)\n    with open(path, "w", newline="") as f:\n        w = csv.writer(f)\n        w.writerow(["SOPInstanceUID", "Target"])\n        for u, row in zip(uids, pred):\n            w.writerow([u, " ".join(str(int(c)) for c in np.where(row == 1)[0])])\n    return pred\n\n\ndef check_submission(path: str, sample_path: str, n_classes: int = N_CLASSES) -> dict:\n    """Validate a submission against Kaggle\'s sample_submission.csv. Raises AssertionError."""\n    with open(path, newline="") as f:\n        sub = list(csv.DictReader(f))\n    with open(sample_path, newline="") as f:\n        ref = list(csv.DictReader(f))\n    assert list(sub[0].keys()) == ["SOPInstanceUID", "Target"], f"bad header {list(sub[0].keys())}"\n    assert len(sub) == len(ref), f"row count {len(sub)} != {len(ref)}"\n    assert {r["SOPInstanceUID"] for r in sub} == {r["SOPInstanceUID"] for r in ref}, "id set differs"\n    assert len({r["SOPInstanceUID"] for r in sub}) == len(sub), "duplicate ids"\n    n_labels = []\n    for r in sub:\n        labs = r["Target"].split()\n        assert len(labs) >= 1, f"empty Target for {r[\'SOPInstanceUID\']}"\n        ints = [int(x) for x in labs]\n        assert all(0 <= i < n_classes for i in ints), f"label out of range: {labs}"\n        assert len(set(ints)) == len(ints), f"duplicate labels: {labs}"\n        n_labels.append(len(ints))\n    return {"rows": len(sub), "mean_labels_per_image": float(np.mean(n_labels)),\n            "frac_single_label": float(np.mean(np.array(n_labels) == 1))}\n')
print("wrote src/evalkit.py")

In [ ]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/train_v2.py", "w", encoding="utf-8") as f:
    f.write('"""\ntrain_v2 -- one training run (one backbone, one fold) designed for CLEAN out-of-fold (OOF)\npredictions.\n\nDifferences from src/train.py / engine.py (and why)\n---------------------------------------------------\n* NO checkpoint selection.  The old engine kept the epoch with the best validation\n  macro_f1 of the SAME fold that was later reported / used to tune thresholds, which makes\n  every reported number optimistic.  Here the schedule is fixed (cosine, N epochs) and the\n  final EMA weights are used.  Per-epoch validation numbers are printed for monitoring only.\n* EMA of the weights (smoother, usually better than the last raw epoch, needs no selection).\n* Two heads on one backbone: a multilabel sigmoid head (22 logits) and a single-label\n  softmax head (22 logits).  ~92 % of UNIFESP images carry exactly one label, and the\n  2nd-place Kaggle solution framed it as single-label multiclass.  Both heads are saved;\n  evalkit/run_ensemble choose / blend them on OOF only.\n* Predictions for OOF fold, held-out set (optional) and the Kaggle test set are written to\n  ``preds.npz`` so the ensemble stage is pure numpy (no GPU).\n* fp16 autocast with logits-space losses only (the V3 crash was BCE on probabilities).\n\nUsage (Kaggle, GPU):\n    python -m src.train_v2 --backbone swin_tiny --img_size 224 --fold 0 \\\n        --folds_csv /kaggle/working/splits/cv_folds.csv --fold_col fold \\\n        --holdout_csv /kaggle/working/splits/holdout_uids.csv --run_name swin_tiny_s224_f0\n\nSmoke test (CPU, a few images, no pretrained weights, no real training):\n    python -m src.train_v2 --backbone resnet18 --img_size 64 --fold 0 --epochs 1 --smoke_test ...\n"""\nimport argparse\nimport copy\nimport json\nimport os\nimport random\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader\n\nfrom . import config as C\nfrom . import evalkit as E\nfrom .dataset import UnifespDataset, build_uid_to_path\n\nN = C.N_CLASSES\nLABEL_COLS = [f"label_{k}_{C.LABEL_NAMES[k].replace(\' \', \'_\')}" for k in range(N)]\n\n# short name -> timm name (anything else is passed to timm unchanged)\nTIMM_NAMES = {\n    "swin_tiny": "swin_tiny_patch4_window7_224",\n    "efficientnetv2_s": "tf_efficientnetv2_s",\n    "densenet121": "densenet121",\n    "convnext_tiny": "convnext_tiny",\n    "convnext_small": "convnext_small",\n    "resnet50": "resnet50",\n    "resnet152": "resnet152",\n    "efficientnet_b3": "efficientnet_b3",\n}\n\n\n# ----------------------------------------------------------------------------- model\nclass TwoHeadNet(nn.Module):\n    """timm backbone (pooled features) + one linear layer producing 2*N logits:\n    [:N] -> multilabel (sigmoid), [N:] -> single-label (softmax)."""\n\n    def __init__(self, backbone, pretrained=True, drop=0.2):\n        super().__init__()\n        import timm\n        name = TIMM_NAMES.get(backbone, backbone)\n        self.body = timm.create_model(name, pretrained=pretrained, num_classes=0)\n        self.drop = nn.Dropout(drop)\n        with torch.no_grad():          # measure the real feature size (timm\'s num_features is wrong for e.g. VGG: 512 vs 4096)\n            was_training = self.body.training\n            feat = self.body.eval()(torch.zeros(1, 3, 224, 224)).shape[1]\n            self.body.train(was_training)\n        self.head = nn.Linear(feat, 2 * N)\n\n    def forward(self, x):\n        z = self.head(self.drop(self.body(x)))\n        return z[:, :N], z[:, N:]\n\n\ndef build_model(name, pretrained=True):\n    """timm backbone (TwoHeadNet) or one of the DenseNet+Swin fusion variants (fusion.py, input 224)."""\n    if name.startswith("hybrid_"):\n        from .fusion import FusionNet\n        return FusionNet(name, pretrained=pretrained)\n    return TwoHeadNet(name, pretrained=pretrained)\n\n\ndef compute_pos_weight(y, cap=20.0):\n    pos = np.clip(y.sum(0), 1, None)\n    return torch.tensor(np.clip((len(y) - pos) / pos, 1.0, cap), dtype=torch.float32)\n\n\nclass TwoHeadLoss(nn.Module):\n    """BCE-with-logits (pos_weight, capped) on the sigmoid head\n       + lam * soft-target cross-entropy on the softmax head (target = y / y.sum())."""\n\n    def __init__(self, pos_weight=None, lam=0.5, smoothing=0.0):\n        super().__init__()\n        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)\n        self.lam, self.smoothing = lam, smoothing\n\n    def forward(self, outs, y):\n        sig, soft = outs\n        loss = self.bce(sig.float(), y)\n        if self.lam > 0:\n            t = y / y.sum(1, keepdim=True).clamp(min=1.0)\n            if self.smoothing > 0:\n                t = t * (1 - self.smoothing) + self.smoothing / N\n            loss = loss + self.lam * (-(t * F.log_softmax(soft.float(), dim=1)).sum(1)).mean()\n        return loss\n\n\nclass EMA:\n    """Exponential moving average of parameters AND buffers (BN statistics)."""\n\n    def __init__(self, model, decay=0.998):\n        self.ema = copy.deepcopy(model).eval()\n        for p in self.ema.parameters():\n            p.requires_grad_(False)\n        self.decay, self.steps = decay, 0\n\n    @torch.no_grad()\n    def update(self, model):\n        self.steps += 1\n        d = min(self.decay, (1 + self.steps) / (10 + self.steps))\n        msd = model.state_dict()\n        for k, v in self.ema.state_dict().items():\n            if v.dtype.is_floating_point:\n                v.mul_(d).add_(msd[k].detach(), alpha=1 - d)\n            else:\n                v.copy_(msd[k])\n\n\n# ----------------------------------------------------------------------------- data\ndef get_transforms(img_size, train, aug="light"):\n    import albumentations as A\n    from albumentations.pytorch import ToTensorV2\n    mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)\n    if not train:\n        return A.Compose([A.Resize(img_size, img_size), A.Normalize(mean=mean, std=std), ToTensorV2()])\n    if aug == "heavy":      # in the spirit of the 2nd-place Kaggle writeup; defaults only (API-stable)\n        ops = [A.Resize(img_size, img_size), A.HorizontalFlip(p=0.5),\n               A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.15, rotate_limit=15, p=0.6),\n               A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.6),\n               A.GaussNoise(p=0.2), A.GaussianBlur(p=0.1)]\n    else:                   # "light": the project\'s earlier default\n        ops = [A.Resize(img_size, img_size), A.HorizontalFlip(p=0.5),\n               A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.5),\n               A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5)]\n    return A.Compose(ops + [A.Normalize(mean=mean, std=std), ToTensorV2()])\n\n\ndef make_frames(args):\n    index_df = pd.read_csv(C.INDEX_CSV)\n    folds = pd.read_csv(args.folds_csv)[["uid", args.fold_col]]\n    tr_all = index_df[index_df["split"] == "train"].merge(folds, on="uid", how="inner")\n    train_df = tr_all[tr_all[args.fold_col] != args.fold].reset_index(drop=True)\n    oof_df = tr_all[tr_all[args.fold_col] == args.fold].reset_index(drop=True)\n    hold_df = None\n    if args.holdout_csv:\n        hold_uids = set(pd.read_csv(args.holdout_csv)["uid"])\n        hold_df = index_df[index_df["uid"].isin(hold_uids)].reset_index(drop=True)\n    test_df = index_df[index_df["split"] == "test"].reset_index(drop=True)\n    if args.smoke_test:\n        train_df, oof_df, test_df = train_df.iloc[:8], oof_df.iloc[:4], test_df.iloc[:4]\n        hold_df = None if hold_df is None else hold_df.iloc[:4]\n    if getattr(args, "pseudo_csv", None):\n        ps = pd.read_csv(args.pseudo_csv)\n        miss = [c for c in LABEL_COLS if c not in ps.columns]\n        assert not miss, f"pseudo csv lacks label columns {miss[:3]}..."\n        if args.smoke_test:\n            ps = ps.iloc[:4]\n        train_df = pd.concat([train_df, ps.reindex(columns=train_df.columns)], ignore_index=True)\n        print(f"[pseudo] +{len(ps)} pseudo-labelled images added to the training part")\n    return train_df, oof_df, hold_df, test_df\n\n\n# ----------------------------------------------------------------------------- predict\n@torch.no_grad()\ndef predict(model, loader, device, tta=False):\n    model.eval()\n    sig_all, soft_all, uids = [], [], []\n    for images, second in loader:\n        images = images.to(device, non_blocking=True)\n        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):\n            s, f = model(images)\n            ps, pf = torch.sigmoid(s.float()), torch.softmax(f.float(), dim=1)\n            if tta:\n                s2, f2 = model(torch.flip(images, dims=[3]))\n                ps = (ps + torch.sigmoid(s2.float())) / 2\n                pf = (pf + torch.softmax(f2.float(), dim=1)) / 2\n        sig_all.append(ps.cpu().numpy()); soft_all.append(pf.cpu().numpy())\n        uids.extend(list(second) if not torch.is_tensor(second) else [None] * len(images))\n    return np.concatenate(sig_all).astype(np.float32), np.concatenate(soft_all).astype(np.float32), uids\n\n\ndef run_predictions(model, df, has_labels, args, device, uid_to_path, img_size):\n    ds = UnifespDataset(df, C.DATA_ROOT, transform=get_transforms(img_size, False), has_labels=False,\n                        uid_to_path=uid_to_path)\n    dl = DataLoader(ds, batch_size=args.batch_size, shuffle=False, num_workers=args.num_workers)\n    out = {}\n    for tag, tta in (("", False), ("_tta", True)):\n        s, f, uids = predict(model, dl, device, tta=tta)\n        out["sig" + tag], out["soft" + tag] = s, f\n    out["uid"] = np.array(uids if uids and uids[0] is not None else df["uid"].tolist())\n    if has_labels:\n        out["y"] = df[LABEL_COLS].to_numpy(dtype=np.int64)\n    return out\n\n\n# ----------------------------------------------------------------------------- main\ndef set_seed(seed):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--backbone", required=True)\n    p.add_argument("--img_size", type=int, default=224)\n    p.add_argument("--fold", type=int, required=True)\n    p.add_argument("--folds_csv", required=True)\n    p.add_argument("--fold_col", default="fold")\n    p.add_argument("--holdout_csv", default=None)\n    p.add_argument("--pseudo_csv", default=None,\n                    help="optional CSV (uid + label_* columns) of pseudo-labelled UNLABELLED images (e.g. Kaggle test images); "\n                         "added to the TRAINING part of every fold only, never to OOF / held-out")\n    p.add_argument("--epochs", type=int, default=30)\n    p.add_argument("--batch_size", type=int, default=32)\n    p.add_argument("--lr", type=float, default=3e-4)\n    p.add_argument("--wd", type=float, default=1e-4)\n    p.add_argument("--warmup_epochs", type=float, default=1.0)\n    p.add_argument("--ema", type=float, default=0.998)\n    p.add_argument("--aug", default="light", choices=["light", "heavy"])\n    p.add_argument("--lam_ce", type=float, default=0.5, help="weight of the softmax single-label head loss (0 = off)")\n    p.add_argument("--smoothing", type=float, default=0.0)\n    p.add_argument("--seed", type=int, default=C.SEED)\n    p.add_argument("--num_workers", type=int, default=4)\n    p.add_argument("--run_name", default=None)\n    p.add_argument("--out_dir", default=C.OUTPUT_DIR)\n    p.add_argument("--smoke_test", action="store_true", help="tiny subset, no pretrained weights (CPU code-path check)")\n    p.add_argument("--no_pretrained", action="store_true", help="random init (local sanity runs only)")\n    args = p.parse_args()\n\n    set_seed(args.seed)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    run_name = args.run_name or f"{args.backbone}_s{args.img_size}_f{args.fold}"\n    run_dir = os.path.join(args.out_dir, run_name)\n    os.makedirs(run_dir, exist_ok=True)\n\n    train_df, oof_df, hold_df, test_df = make_frames(args)\n    print(f"[{run_name}] train={len(train_df)} oof={len(oof_df)} "\n          f"holdout={0 if hold_df is None else len(hold_df)} test={len(test_df)} device={device}")\n    uid_to_path = build_uid_to_path(C.DATA_ROOT)\n\n    tr_ds = UnifespDataset(train_df, C.DATA_ROOT, transform=get_transforms(args.img_size, True, args.aug),\n                           uid_to_path=uid_to_path)\n    va_ds = UnifespDataset(oof_df, C.DATA_ROOT, transform=get_transforms(args.img_size, False),\n                           uid_to_path=uid_to_path)\n    tr_dl = DataLoader(tr_ds, batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers,\n                       pin_memory=device.type == "cuda", drop_last=len(train_df) > args.batch_size)\n    va_dl = DataLoader(va_ds, batch_size=args.batch_size, shuffle=False, num_workers=args.num_workers)\n\n    model = build_model(args.backbone, pretrained=not (args.smoke_test or args.no_pretrained)).to(device)\n    ema = EMA(model, args.ema)\n    loss_fn = TwoHeadLoss(compute_pos_weight(train_df[LABEL_COLS].to_numpy(np.float32)).to(device),\n                          lam=args.lam_ce, smoothing=args.smoothing)\n    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.wd)\n    steps_per_epoch = max(1, len(tr_dl))\n    total, warm = args.epochs * steps_per_epoch, int(args.warmup_epochs * steps_per_epoch)\n    sched = torch.optim.lr_scheduler.LambdaLR(\n        opt, lambda s: (s + 1) / max(1, warm) if s < warm else 0.5 * (1 + np.cos(np.pi * (s - warm) / max(1, total - warm))))\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")\n\n    t0, history = time.time(), []\n    for epoch in range(args.epochs):\n        model.train(); run_loss = 0.0\n        for images, y in tr_dl:\n            images, y = images.to(device, non_blocking=True), y.to(device, non_blocking=True)\n            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):\n                outs = model(images)\n            loss = loss_fn(outs, y)\n            opt.zero_grad(set_to_none=True)\n            scaler.scale(loss).backward()\n            scaler.unscale_(opt)\n            nn.utils.clip_grad_norm_(model.parameters(), 5.0)\n            scaler.step(opt); scaler.update(); sched.step(); ema.update(model)\n            run_loss += float(loss.detach())\n        # monitoring only -- NEVER used to pick a checkpoint\n        s, _, _ = predict(ema.ema, va_dl, device)\n        yv = oof_df[LABEL_COLS].to_numpy(np.int64)\n        mon = E.sample_mean_f1(yv, E.predict_labels(s, 0.5, min_one=True))\n        history.append({"epoch": epoch, "train_loss": run_loss / steps_per_epoch, "oof_sample_mean_f1_monitor": mon})\n        print(f"epoch {epoch}: loss={run_loss / steps_per_epoch:.4f} monitor_oof_f1={mon:.4f} ({time.time() - t0:.0f}s)")\n\n    # final EMA weights -> predictions (no selection)\n    preds = {}\n    for tag, df, has_y in (("oof", oof_df, True), ("hold", hold_df, True), ("test", test_df, False)):\n        if df is None or len(df) == 0:\n            continue\n        r = run_predictions(ema.ema, df, has_y, args, device, uid_to_path, args.img_size)\n        for k, v in r.items():\n            preds[f"{tag}_{k}"] = v\n    np.savez_compressed(os.path.join(run_dir, "preds.npz"), **preds)\n    torch.save(ema.ema.state_dict(), os.path.join(run_dir, "final_ema.pt"))   # state_dict only (weights_only-safe)\n    meta = {"args": vars(args), "history": history, "seconds": time.time() - t0, "torch": torch.__version__}\n    if hasattr(ema.ema, "gate_weights"):      # fusion variants: how much weight does the CNN branch get?\n        ws = []\n        with torch.no_grad():\n            for images, _ in va_dl:\n                ws.append(ema.ema.gate_weights(images.to(device))[:, 0].float().cpu())\n        w = torch.cat(ws)\n        meta["gate_cnn_weight_oof_mean"], meta["gate_cnn_weight_oof_std"] = float(w.mean()), float(w.std())\n        print(f"[{run_name}] mean CNN-branch gate weight on OOF fold: {float(w.mean()):.4f}")\n    json.dump(meta, open(os.path.join(run_dir, "meta.json"), "w"), indent=1, default=str)\n    print(f"[{run_name}] done in {time.time() - t0:.0f}s -> {run_dir}")\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote src/train_v2.py")

In [ ]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/run_ensemble.py", "w", encoding="utf-8") as f:
    f.write('"""\nrun_ensemble -- CPU-only stage that turns the per-fold ``preds.npz`` files written by\ntrain_v2 into (a) honest numbers and (b) a Kaggle submission.\n\nHonesty rules enforced here\n---------------------------\n* Every DECISION (which head / TTA variant per run, ensemble weights, per-class thresholds)\n  is made on the out-of-fold (OOF) predictions of the CV pool ONLY.\n* The held-out set is touched exactly once, at the end, to REPORT.\n* Held-out and Kaggle-test probabilities of a run are the average of its 5 fold models\n  (bagging), which is how the Kaggle submission is produced too.\n\nInputs:  <out_dir>/<run>_f<k>/preds.npz  for every run prefix and fold k.\nOutputs: <result_dir>/results.json, per_run.csv, ensemble_arrays.npz, submission.csv\n"""\nimport argparse\nimport json\nimport os\n\nimport numpy as np\nimport pandas as pd\n\nfrom . import evalkit as E\n\nN = 22\n\n\ndef _load_run(out_dir, run, folds, pool_uids, with_hold, with_test, n_classes=N):\n    pos = {u: i for i, u in enumerate(pool_uids)}\n    keys = ["sig", "soft", "sig_tta", "soft_tta"]\n    oof = {k: np.full((len(pool_uids), n_classes), np.nan, np.float32) for k in keys}\n    hold = {k: [] for k in keys}\n    test = {k: [] for k in keys}\n    hold_uid = test_uid = None\n    for f in folds:\n        d = np.load(os.path.join(out_dir, f"{run}_f{f}", "preds.npz"), allow_pickle=False)\n        idx = np.array([pos[u] for u in d["oof_uid"]])\n        for k in keys:\n            oof[k][idx] = d[f"oof_{k}"]\n        if with_hold and "hold_uid" in d.files:\n            hu = d["hold_uid"]\n            hold_uid = hu if hold_uid is None else hold_uid\n            assert (hu == hold_uid).all(), "holdout uid order differs between folds"\n            for k in keys:\n                hold[k].append(d[f"hold_{k}"])\n        if with_test and "test_uid" in d.files:\n            tu = d["test_uid"]\n            test_uid = tu if test_uid is None else test_uid\n            assert (tu == test_uid).all(), "test uid order differs between folds"\n            for k in keys:\n                test[k].append(d[f"test_{k}"])\n    for k in keys:\n        assert not np.isnan(oof[k]).any(), f"{run}: OOF not complete for {k} (folds={folds})"\n    avg = lambda L: {k: np.mean(L[k], axis=0) for k in keys} if L[keys[0]] else None\n    return {"oof": oof, "hold": avg(hold), "test": avg(test), "hold_uid": hold_uid, "test_uid": test_uid}\n\n\ndef _variants(p):\n    """Candidate probability matrices for one run (all selected on OOF later)."""\n    v = {"sig": p["sig"], "soft": p["soft"], "blend": E.combine_heads(p["sig"], p["soft"], 0.5),\n         "sig_tta": p["sig_tta"], "soft_tta": p["soft_tta"],\n         "blend_tta": E.combine_heads(p["sig_tta"], p["soft_tta"], 0.5)}\n    return v\n\n\ndef run(args):\n    os.makedirs(args.result_dir, exist_ok=True)\n    index = pd.read_csv(args.index_csv)\n    label_cols = [c for c in index.columns if c.startswith("label_")]\n    n_classes = len(label_cols)             # 22 for UNIFESP, 7 MURA, 4 COVID, ...\n    pool = pd.read_csv(args.cv_folds_csv)\n    pool_uids = pool["uid"].tolist()\n    lab = index.set_index("uid")\n    y_pool = lab.loc[pool_uids, label_cols].to_numpy(np.int64)\n\n    have_hold = bool(args.holdout_csv)\n    hold_uids = pd.read_csv(args.holdout_csv)["uid"].tolist() if have_hold else []\n    folds = [int(x) for x in args.folds.split(",")]\n    runs = [r for r in args.runs.split(",") if r]\n\n    data, chosen, rows = {}, {}, []\n    for r in runs:\n        d = _load_run(args.out_dir, r, folds, pool_uids, have_hold, True, n_classes)\n        vs_oof = _variants(d["oof"])\n        scores = {k: E.sample_mean_f1(y_pool, E.predict_labels(v, 0.5, min_one=True)) for k, v in vs_oof.items()}\n        best = max(scores, key=scores.get)                     # decision on OOF only\n        chosen[r] = best\n        data[r] = {"oof": vs_oof[best],\n                   "hold": _variants(d["hold"])[best] if d["hold"] is not None else None,\n                   "test": _variants(d["test"])[best] if d["test"] is not None else None,\n                   "hold_uid": d["hold_uid"], "test_uid": d["test_uid"]}\n        rows.append({"run": r, "chosen_variant": best, **{f"oof_{k}": s for k, s in scores.items()}})\n\n    # ---- ensemble weights on OOF only\n    oof_list = [data[r]["oof"] for r in runs]\n    w = E.caruana_ensemble(oof_list, y_pool, n_iter=args.caruana_iters)\n    ens_oof = E.average_probs(oof_list, w)\n    thr = E.tune_thresholds_oof(ens_oof, y_pool, min_pos=args.min_pos)\n\n    result = {"runs": runs, "chosen_variant": chosen, "weights": dict(zip(runs, map(float, w))),\n              "thresholds": [float(t) for t in thr],\n              "oof": {"ens_0.5_min_one": E.all_metrics(y_pool, E.predict_labels(ens_oof, 0.5, True)),\n                      "note": "OOF numbers are for model selection only; report the held-out numbers"}}\n\n    # ---- held-out set: used once, to report\n    if have_hold:\n        hold_uid_order = data[runs[0]]["hold_uid"].tolist()\n        y_hold = lab.loc[hold_uid_order, label_cols].to_numpy(np.int64)\n        groups = lab.loc[hold_uid_order, "patient_id"].astype(str).to_numpy()\n        hold_list = [data[r]["hold"] for r in runs]\n        ens_hold = E.average_probs(hold_list, w)\n        pred_05 = E.predict_labels(ens_hold, 0.5, min_one=False)\n        pred_05m = E.predict_labels(ens_hold, 0.5, min_one=True)\n        pred_tuned = E.predict_labels(ens_hold, thr, min_one=True)\n        result["holdout"] = {"n_images": len(y_hold), "n_patients": int(len(set(groups))),\n                             "ens_0.5": E.all_metrics(y_hold, pred_05),\n                             "ens_0.5_min_one": E.all_metrics(y_hold, pred_05m),\n                             "ens_tuned_min_one": E.all_metrics(y_hold, pred_tuned)}\n        result["holdout"]["ci_sample_mean_f1_tuned"] = E.cluster_bootstrap(\n            E.sample_mean_f1, y_hold, pred_tuned, groups, n_boot=args.n_boot)\n        result["holdout"]["ci_macro_f1_tuned"] = E.cluster_bootstrap(\n            E.macro_f1, y_hold, pred_tuned, groups, n_boot=args.n_boot)\n        # best single run (chosen on OOF!) vs ensemble, paired on the same patient resamples\n        single_oof = {r: E.sample_mean_f1(y_pool, E.predict_labels(data[r]["oof"], 0.5, True)) for r in runs}\n        best_single = max(single_oof, key=single_oof.get)\n        pred_single = E.predict_labels(data[best_single]["hold"], 0.5, min_one=True)\n        result["holdout"]["best_single_by_oof"] = best_single\n        result["holdout"]["best_single_metrics"] = E.all_metrics(y_hold, pred_single)\n        result["holdout"]["paired_ens_vs_single_sample_f1"] = E.paired_bootstrap(\n            E.sample_mean_f1, y_hold, pred_tuned, pred_single, groups, n_boot=args.n_boot)\n        for row in rows:\n            r = row["run"]\n            row["holdout_sample_mean_f1_0.5_min_one"] = E.sample_mean_f1(\n                y_hold, E.predict_labels(data[r]["hold"], 0.5, min_one=True))\n\n    # ---- Kaggle test submission (UNIFESP only: other datasets have no hidden test set)\n    ens_test = None\n    if data[runs[0]]["test"] is not None:\n        test_uid_order = data[runs[0]]["test_uid"].tolist()\n        ens_test = E.average_probs([data[r]["test"] for r in runs], w)\n        sub_path = os.path.join(args.result_dir, "submission.csv")\n        pred = E.write_submission(test_uid_order, ens_test, thr, sub_path, min_one=True)\n        result["submission"] = {"path": sub_path, "mean_labels": float(pred.sum(1).mean())}\n        if args.sample_submission:\n            result["submission"]["check"] = E.check_submission(sub_path, args.sample_submission, n_classes)\n\n    pd.DataFrame(rows).to_csv(os.path.join(args.result_dir, "per_run.csv"), index=False)\n    json.dump(result, open(os.path.join(args.result_dir, "results.json"), "w"), indent=1, default=float)\n    np.savez_compressed(os.path.join(args.result_dir, "ensemble_arrays.npz"), oof=ens_oof,\n                        hold=ens_hold if have_hold else np.zeros(0),\n                        test=ens_test if ens_test is not None else np.zeros(0), thresholds=thr, weights=w,\n                        hold_uid=np.array(data[runs[0]]["hold_uid"]) if have_hold else np.zeros(0),\n                        test_uid=np.array(data[runs[0]]["test_uid"]) if ens_test is not None else np.zeros(0),\n                        pool_uid=np.array(pool_uids))\n    return result\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--out_dir", required=True, help="folder that contains <run>_f<k>/preds.npz")\n    p.add_argument("--runs", required=True, help="comma-separated run prefixes, e.g. swin_tiny_s224,densenet121_s256")\n    p.add_argument("--folds", default="0,1,2,3,4")\n    p.add_argument("--index_csv", required=True)\n    p.add_argument("--cv_folds_csv", required=True)\n    p.add_argument("--holdout_csv", default=None)\n    p.add_argument("--sample_submission", default=None)\n    p.add_argument("--result_dir", required=True)\n    p.add_argument("--n_boot", type=int, default=2000)\n    p.add_argument("--caruana_iters", type=int, default=40)\n    p.add_argument("--min_pos", type=float, default=20.0)\n    res = run(p.parse_args())\n    print(json.dumps({k: res[k] for k in ("chosen_variant", "weights", "holdout") if k in res}, indent=1, default=float))\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote src/run_ensemble.py")

In [ ]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/merge_runs.py", "w", encoding="utf-8") as f:
    f.write('"""\nmerge_runs -- helpers for running the pipeline as several notebooks (same Kaggle account) and merging.\n\nWhy a hash check: every part notebook rebuilds index.csv / splits from the same code and seed, so\nthey SHOULD be identical.  If they are not (different dataset version, library version, ...) the\nOOF predictions of the parts refer to different folds and must not be merged.\n\n    python -m src.merge_runs --write_hashes --index_csv index.csv --splits_dir splits --out split_hashes.json\n    python -m src.merge_runs --input_root /kaggle/input --out_dir runs --index_csv index.csv --splits_dir splits\n"""\nimport argparse\nimport hashlib\nimport json\nimport os\nimport re\nimport shutil\n\nRUN_DIR = re.compile(r"^(?P<prefix>.+)_f(?P<fold>\\d+)$")\nHASHED = ("cv_folds.csv", "holdout_uids.csv", "folds.csv")\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for chunk in iter(lambda: f.read(1 << 20), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef split_hashes(index_csv, splits_dir):\n    out = {"index.csv": sha256(index_csv)}\n    for n in HASHED:\n        p = os.path.join(splits_dir, n)\n        if os.path.exists(p):\n            out[n] = sha256(p)\n    return out\n\n\ndef discover(input_root):\n    """{prefix: {fold: dir}} for every <prefix>_f<k>/preds.npz found anywhere under input_root."""\n    found = {}\n    for root, dirs, files in os.walk(input_root):\n        if "preds.npz" in files:\n            m = RUN_DIR.match(os.path.basename(root))\n            if m:\n                found.setdefault(m["prefix"], {}).setdefault(int(m["fold"]), root)\n    return found\n\n\ndef check_hashes(input_root, local):\n    """Compare local split hashes with every split_hashes.json found under input_root."""\n    problems, seen = [], 0\n    for root, _d, files in os.walk(input_root):\n        if "split_hashes.json" in files:\n            seen += 1\n            theirs = json.load(open(os.path.join(root, "split_hashes.json")))\n            for k, v in theirs.items():\n                if k in local and local[k] != v:\n                    problems.append(f"{root}: {k} differs")\n    return seen, problems\n\n\ndef collect(input_root, out_dir, folds=(0, 1, 2, 3, 4)):\n    os.makedirs(out_dir, exist_ok=True)\n    found = discover(input_root)\n    complete, incomplete = [], {}\n    for prefix, by_fold in sorted(found.items()):\n        missing = [f for f in folds if f not in by_fold]\n        if missing:\n            incomplete[prefix] = missing\n            continue\n        for f, src in by_fold.items():\n            dst = os.path.join(out_dir, f"{prefix}_f{f}")\n            os.makedirs(dst, exist_ok=True)\n            for name in ("preds.npz", "meta.json"):\n                if os.path.exists(os.path.join(src, name)):\n                    shutil.copy2(os.path.join(src, name), os.path.join(dst, name))\n        complete.append(prefix)\n    return complete, incomplete\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--write_hashes", action="store_true")\n    p.add_argument("--index_csv", required=True)\n    p.add_argument("--splits_dir", required=True)\n    p.add_argument("--out", default="split_hashes.json")\n    p.add_argument("--input_root", default="/kaggle/input")\n    p.add_argument("--out_dir", default="runs")\n    p.add_argument("--folds", default="0,1,2,3,4")\n    p.add_argument("--allow_hash_mismatch", action="store_true")\n    a = p.parse_args()\n    local = split_hashes(a.index_csv, a.splits_dir)\n    if a.write_hashes:\n        json.dump(local, open(a.out, "w"), indent=1)\n        print("wrote", a.out, local)\n        return\n    seen, problems = check_hashes(a.input_root, local)\n    print(f"split_hashes.json files found: {seen}")\n    if problems and not a.allow_hash_mismatch:\n        raise SystemExit("SPLIT MISMATCH -- do not merge:\\n  " + "\\n  ".join(problems))\n    complete, incomplete = collect(a.input_root, a.out_dir, tuple(int(x) for x in a.folds.split(",")))\n    print("complete runs :", complete)\n    print("incomplete    :", incomplete, "(skipped)")\n    if not complete:\n        raise SystemExit("no complete run found under " + a.input_root)\n    open(os.path.join(a.out_dir, "RUNS.txt"), "w").write(",".join(complete))\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote src/merge_runs.py")

In [ ]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/fusion.py", "w", encoding="utf-8") as f:
    f.write('"""\nfusion -- DenseNet121 (CNN) + Swin-Tiny (Transformer) fusion variants for the gate-collapse ablation.\n\nBackground: in the earlier notebooks the learned gate put ~0.0001 weight on the CNN branch in 5/5 CV folds\n(i.e. it silently became a plain Swin-Tiny).  Hypothesis (UNTESTED): the gate sees raw pooled features whose scales\ndiffer (DenseNet: post-ReLU average-pool; Swin: LayerNorm output), so the softmax saturates.  Variants:\n\n    hybrid_gate     original: gate on raw concatenated features             (reproduces the collapse)\n    hybrid_gate_ln  LayerNorm both feature vectors before gate AND projections (the proposed fix)\n    hybrid_avg      fixed 0.5 / 0.5 average of the projected branches        (no gate at all)\n    hybrid_cnn      CNN branch only (same projection + head)\n    hybrid_swin     Swin branch only (same projection + head)\n\nAll variants return (sigmoid_logits, softmax_logits) like TwoHeadNet, so train_v2 / run_ensemble need no change.\n``gate_weights(x)`` returns the (B, 2) branch weights [cnn, swin] for the analysis.\nInput size must be 224 (Swin-Tiny window7_224).\n"""\nimport torch\nimport torch.nn as nn\n\nfrom . import config as C\n\nN = C.N_CLASSES\nVARIANTS = ("hybrid_gate", "hybrid_gate_ln", "hybrid_avg", "hybrid_cnn", "hybrid_swin")\n\n\nclass FusionNet(nn.Module):\n    def __init__(self, variant, pretrained=True, fusion_dim=512, drop=0.2,\n                 cnn_name="densenet121", swin_name="swin_tiny_patch4_window7_224"):\n        super().__init__()\n        assert variant in VARIANTS, f"unknown fusion variant {variant}; options {VARIANTS}"\n        import timm\n        self.variant = variant\n        self.cnn = None if variant == "hybrid_swin" else timm.create_model(cnn_name, pretrained=pretrained, num_classes=0)\n        self.swin = None if variant == "hybrid_cnn" else timm.create_model(swin_name, pretrained=pretrained, num_classes=0)\n        cd = self.cnn.num_features if self.cnn is not None else 0\n        sd = self.swin.num_features if self.swin is not None else 0\n        self.ln_c = nn.LayerNorm(cd) if (variant == "hybrid_gate_ln" and cd) else None\n        self.ln_s = nn.LayerNorm(sd) if (variant == "hybrid_gate_ln" and sd) else None\n        self.proj_c = nn.Linear(cd, fusion_dim) if cd else None\n        self.proj_s = nn.Linear(sd, fusion_dim) if sd else None\n        self.gate = (nn.Sequential(nn.Linear(cd + sd, 128), nn.ReLU(inplace=True), nn.Linear(128, 2))\n                     if variant in ("hybrid_gate", "hybrid_gate_ln") else None)\n        self.drop = nn.Dropout(drop)\n        self.head = nn.Linear(fusion_dim, 2 * N)\n\n    def _features(self, x):\n        fc = self.cnn(x) if self.cnn is not None else None\n        fs = self.swin(x) if self.swin is not None else None\n        if self.ln_c is not None:\n            fc, fs = self.ln_c(fc), self.ln_s(fs)\n        return fc, fs\n\n    def _weights(self, fc, fs):\n        ref = fc if fc is not None else fs\n        B = ref.size(0)\n        if self.variant == "hybrid_avg":\n            return ref.new_full((B, 2), 0.5)\n        if self.variant == "hybrid_cnn":\n            return ref.new_tensor([1.0, 0.0]).expand(B, 2)\n        if self.variant == "hybrid_swin":\n            return ref.new_tensor([0.0, 1.0]).expand(B, 2)\n        return torch.softmax(self.gate(torch.cat([fc, fs], dim=1)), dim=1)\n\n    def forward(self, x):\n        fc, fs = self._features(x)\n        w = self._weights(fc, fs)\n        fused = 0\n        if fc is not None:\n            fused = fused + w[:, 0:1] * self.proj_c(fc)\n        if fs is not None:\n            fused = fused + w[:, 1:2] * self.proj_s(fs)\n        z = self.head(self.drop(fused))\n        return z[:, :N], z[:, N:]\n\n    @torch.no_grad()\n    def gate_weights(self, x):\n        fc, fs = self._features(x)\n        return self._weights(fc, fs)\n')
print("wrote src/fusion.py")

In [ ]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/cross_eval.py", "w", encoding="utf-8") as f:
    f.write('"""\ncross_eval -- zero-shot transfer of body-part models between public datasets through an explicit class map.\n\n    UNIFESP model  --> MURA images        (6 shared classes; MURA "Humerus" has no UNIFESP counterpart and is dropped)\n    MURA model     --> UNIFESP images     (single-label images of the 6 shared classes)\n    either model   --> FracAtlas images   (hand / shoulder / hip / "leg" group; images with several regions are dropped)\n\nPredictions are restricted to the groups present in BOTH taxonomies and renormalised, i.e. a closed-set decision among\nthe shared groups.  The class map is an ASSUMPTION to be checked by a radiologist (especially FracAtlas "leg", which can\ncover lower leg / knee / ankle); edit ``GROUPS_*`` or pass --map_json.  FracAtlas column names are NOT verified: the\nloader prints the columns it finds and fails loudly if the region columns are missing.\n\nInputs per source run: <weights_dir>/<run_prefix>_f<k>/final_ema.pt (written by train_v2), averaged over folds (bagging).\n"""\nimport argparse\nimport json\nimport os\nimport re\n\nimport numpy as np\n\nfrom . import evalkit as E\n\n# ------------------------------------------------------------------ taxonomy -> shared group (assumption!)\nGROUPS_UNIFESP = {"Elbow": "elbow", "Finger": "finger", "Forearm": "forearm", "Hand": "hand", "Shoulder": "shoulder",\n                  "Wrist": "wrist", "Hip": "hip", "Lower Leg": "leg", "Knee": "leg", "Thigh": "leg"}\nGROUPS_MURA = {"Elbow": "elbow", "Finger": "finger", "Forearm": "forearm", "Hand": "hand", "Shoulder": "shoulder",\n               "Wrist": "wrist"}                                    # Humerus: no counterpart\nGROUPS_FRACATLAS = {"hand": "hand", "leg": "leg", "hip": "hip", "shoulder": "shoulder"}\nGROUPS = {"unifesp": GROUPS_UNIFESP, "mura": GROUPS_MURA, "fracatlas": GROUPS_FRACATLAS}\n\n\ndef class_names_from_columns(cols):\n    """label_3_Chest -> \'Chest\' ; label_12_Lower_Leg -> \'Lower Leg\' (index order preserved)."""\n    out = {}\n    for c in cols:\n        m = re.match(r"label_(\\d+)_(.*)$", c)\n        if m:\n            out[int(m[1])] = m[2].replace("_", " ")\n    return [out[k] for k in sorted(out)]\n\n\ndef shared_groups(src_map, tgt_map):\n    return sorted(set(src_map.values()) & set(tgt_map.values()))\n\n\ndef group_probabilities(prob, class_names, class_to_group, groups, how="sum"):\n    """(N, n_classes) -> (N, n_groups): sum (softmax head) or max (sigmoid head) over the classes of each group, renormalised."""\n    out = np.zeros((prob.shape[0], len(groups)))\n    for gi, g in enumerate(groups):\n        idx = [i for i, n in enumerate(class_names) if class_to_group.get(n) == g]\n        if idx:\n            out[:, gi] = prob[:, idx].sum(1) if how == "sum" else prob[:, idx].max(1)\n    s = out.sum(1, keepdims=True)\n    return out / np.where(s > 0, s, 1.0)\n\n\ndef transfer_metrics(prob_groups, y_group, groups):\n    """Closed-set accuracy / macro-F1 / per-group recall + confusion matrix over the shared groups."""\n    pred = prob_groups.argmax(1)\n    acc = float((pred == y_group).mean())\n    f1s, rec = [], {}\n    for gi, g in enumerate(groups):\n        tp = int(((pred == gi) & (y_group == gi)).sum()); fp = int(((pred == gi) & (y_group != gi)).sum())\n        fn = int(((pred != gi) & (y_group == gi)).sum())\n        den = 2 * tp + fp + fn\n        f1s.append(0.0 if den == 0 else 2 * tp / den)\n        rec[g] = float(tp / (tp + fn)) if (tp + fn) else float("nan")\n    cm = np.zeros((len(groups), len(groups)), dtype=int)\n    for t, p in zip(y_group, pred):\n        cm[t, p] += 1\n    return {"n_images": int(len(y_group)), "accuracy": acc, "macro_f1": float(np.mean(f1s)),\n            "per_group_recall": rec, "groups": list(groups), "confusion": cm.tolist()}\n\n\n# ------------------------------------------------------------------ target datasets\ndef _stem_uid(path):                    # FracAtlas: IMG0000019.jpg -> IMG0000019\n    return os.path.splitext(os.path.basename(path))[0]\n\n\ndef build_path_map(root, kind):\n    """uid -> file path for a target dataset (independent of the global config)."""\n    from .dataset import _basename_uid\n    m = {}\n    for cur, _d, files in os.walk(root):\n        for f in files:\n            if f.startswith("._"):\n                continue\n            low = f.lower()\n            p = os.path.join(cur, f)\n            if kind == "unifesp" and low.endswith(".dcm"):\n                m[_basename_uid(f)] = p\n            elif kind == "mura" and low.endswith(".png"):\n                m[os.path.relpath(p, root).replace(os.sep, "__")[:-4]] = p\n            elif kind == "fracatlas" and low.endswith((".jpg", ".jpeg", ".png")):\n                m[_stem_uid(p)] = p\n    return m\n\n\nMURA_RE = re.compile(r"XR_([A-Z]+)")\n\n\ndef target_items(kind, root, index_csv=None, fracatlas_csv=None, uids_csv=None):\n    """Returns (uids, paths, class_names_of_target_taxonomy, labels) with ONE label (class name) per image, dropping\n    multi-label / unmapped images. ``uids_csv`` (optional, column uid) restricts to a given subset (e.g. the held-out set)."""\n    import pandas as pd\n    paths = build_path_map(root, kind)\n    if kind == "mura":\n        items = []\n        for uid, p in paths.items():\n            m = MURA_RE.search(p.upper())\n            if m:\n                items.append((uid, p, m[1].title()))\n    elif kind == "unifesp":\n        df = pd.read_csv(index_csv)\n        df = df[df["split"] == "train"]\n        cols = [c for c in df.columns if c.startswith("label_")]\n        names = class_names_from_columns(cols)\n        single = df[df[cols].sum(1) == 1]\n        items = [(r["uid"], paths[r["uid"]], names[int(np.argmax(r[cols].to_numpy(float)))])\n                 for _, r in single.iterrows() if r["uid"] in paths]\n    elif kind == "fracatlas":\n        df = pd.read_csv(fracatlas_csv)\n        low = {c.lower(): c for c in df.columns}\n        need = ["hand", "leg", "hip", "shoulder"]\n        missing = [c for c in need if c not in low]\n        if missing or "image_id" not in low:\n            raise SystemExit(f"FracAtlas csv columns not as expected (need image_id + {need}); found {list(df.columns)}")\n        reg = df[[low[c] for c in need]].to_numpy(int)\n        items = []\n        for (_, r), flags in zip(df.iterrows(), reg):\n            if flags.sum() != 1:\n                continue\n            uid = _stem_uid(str(r[low["image_id"]]))\n            if uid in paths:\n                items.append((uid, paths[uid], need[int(flags.argmax())]))\n    else:\n        raise ValueError(kind)\n    if uids_csv:\n        keep = set(pd.read_csv(uids_csv)["uid"])\n        items = [it for it in items if it[0] in keep]\n    uids, ps, labs = zip(*items) if items else ([], [], [])\n    return list(uids), list(ps), list(labs)\n\n\n# ------------------------------------------------------------------ inference\ndef predict_images(paths, weights, backbone, img_size, device, batch=32, minmax=True, tta=False):\n    """Average softmax / sigmoid probabilities of several state_dicts (fold models) on a list of image paths.\n    Every image is decoded ONCE and passed through all models."""\n    import torch\n    from .dataset import load_dicom_as_image, load_png_as_image\n    from .train_v2 import build_model, get_transforms\n    tf = get_transforms(img_size, train=False)\n\n    def load(p):\n        img = load_dicom_as_image(p) if p.lower().endswith(".dcm") else load_png_as_image(p)\n        if minmax and not p.lower().endswith(".dcm"):                      # same per-image min-max as the DICOM loader\n            a = img.astype(np.float32); lo, hi = a.min(), a.max()\n            img = (((a - lo) / (hi - lo + 1e-6)) * 255).astype(np.uint8)\n        return tf(image=img)["image"]\n\n    models = []\n    for w in weights:\n        m = build_model(backbone, pretrained=False)\n        m.load_state_dict(torch.load(w, map_location="cpu", weights_only=True))\n        models.append(m.to(device).eval())\n    soft, sig = [], []\n    with torch.no_grad():\n        for i in range(0, len(paths), batch):\n            x = torch.stack([load(p) for p in paths[i:i + batch]]).to(device)\n            ps_all, pf_all = 0.0, 0.0\n            for m in models:\n                s_, f_ = m(x)\n                ps, pf = torch.sigmoid(s_.float()), torch.softmax(f_.float(), 1)\n                if tta:\n                    s2, f2 = m(torch.flip(x, dims=[3]))\n                    ps, pf = (ps + torch.sigmoid(s2.float())) / 2, (pf + torch.softmax(f2.float(), 1)) / 2\n                ps_all, pf_all = ps_all + ps, pf_all + pf\n            sig.append((ps_all / len(models)).cpu().numpy()); soft.append((pf_all / len(models)).cpu().numpy())\n    return np.concatenate(soft), np.concatenate(sig)\n\n\ndef run(a):\n    import torch\n    os.environ.setdefault("UNIFESP_DATASET", a.src_dataset)\n    from . import config as C\n    src_names = [C.LABEL_NAMES[k] for k in sorted(C.LABEL_NAMES)]\n    src_map = json.load(open(a.map_json))[a.src_dataset] if a.map_json else GROUPS[a.src_dataset]\n    tgt_map = json.load(open(a.map_json))[a.target] if a.map_json else GROUPS[a.target]\n    groups = shared_groups(src_map, tgt_map)\n    uids, paths, labs = target_items(a.target, a.target_root, a.target_index, a.fracatlas_csv, a.target_uids_csv)\n    keep = [i for i, l in enumerate(labs) if tgt_map.get(l) in groups]\n    if a.max_images and len(keep) > a.max_images:                       # reproducible random subsample (speed)\n        keep = sorted(np.random.default_rng(0).choice(keep, size=a.max_images, replace=False).tolist())\n    paths = [paths[i] for i in keep]; uids = [uids[i] for i in keep]\n    y = np.array([groups.index(tgt_map[labs[i]]) for i in keep])\n    print(f"[{a.src_dataset} -> {a.target}] shared groups {groups}; {len(y)} target images "\n          f"(of {len(labs)} single-label); per-group n = {dict(zip(groups, np.bincount(y, minlength=len(groups)).tolist()))}")\n    folds = [int(f) for f in a.folds.split(",")]\n    weights = [os.path.join(a.weights_dir, f"{a.run_prefix}_f{f}", "final_ema.pt") for f in folds]\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    soft, sig = predict_images(paths, weights, a.backbone, a.img_size, device, a.batch, tta=a.tta)\n    out = {"source": a.src_dataset, "target": a.target, "run": a.run_prefix, "groups": groups}\n    for head, p, how in (("softmax", soft, "sum"), ("sigmoid", sig, "max")):\n        pg = group_probabilities(p, src_names, src_map, groups, how)\n        out[head] = transfer_metrics(pg, y, groups)\n        # patient/image bootstrap is image-level here (no patient ids for the cross-dataset targets) -- say so in the paper\n        pred = pg.argmax(1)\n        ones = np.ones((len(y), 1), int)\n        out[head]["accuracy_ci_image_bootstrap"] = E.cluster_bootstrap(\n            lambda yt, yp: float((yt[:, 0] == yp[:, 0]).mean()), y[:, None], pred[:, None], np.arange(len(y)),\n            n_boot=a.n_boot)\n    os.makedirs(a.out_dir, exist_ok=True)\n    path = os.path.join(a.out_dir, f"transfer_{a.src_dataset}_to_{a.target}_{a.run_prefix}.json")\n    json.dump(out, open(path, "w"), indent=1)\n    print(json.dumps({k: {m: out[k][m] for m in ("n_images", "accuracy", "macro_f1")} for k in ("softmax", "sigmoid")}, indent=1))\n    print("saved", path)\n    return out\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--src_dataset", required=True, choices=["unifesp", "mura"])\n    p.add_argument("--target", required=True, choices=["unifesp", "mura", "fracatlas"])\n    p.add_argument("--target_root", required=True)\n    p.add_argument("--target_index", default=None, help="index.csv of the UNIFESP target")\n    p.add_argument("--fracatlas_csv", default=None)\n    p.add_argument("--target_uids_csv", default=None, help="optional: restrict to these uids (e.g. holdout_uids.csv)")\n    p.add_argument("--weights_dir", required=True)\n    p.add_argument("--run_prefix", required=True)\n    p.add_argument("--backbone", required=True)\n    p.add_argument("--img_size", type=int, default=224)\n    p.add_argument("--folds", default="0,1,2,3,4")\n    p.add_argument("--batch", type=int, default=32)\n    p.add_argument("--tta", action="store_true")\n    p.add_argument("--map_json", default=None)\n    p.add_argument("--n_boot", type=int, default=1000)\n    p.add_argument("--max_images", type=int, default=0, help="0 = all target images; else random subsample of this size")\n    p.add_argument("--out_dir", default="transfer_results")\n    run(p.parse_args())\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote src/cross_eval.py")

In [ ]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/ablation_report.py", "w", encoding="utf-8") as f:
    f.write('"""\nablation_report -- turns several finished configurations (each = one or more run prefixes under --out_dir) into\n\n  1. a CONFIG table: OOF F1, held-out F1 (+ patient-bootstrap CI), spread over the 5 fold models, and the paired\n     difference to a reference configuration (CI + p-value)             -> augmentation / loss / backbone / fusion ablations\n  2. a DECODING table for one configuration: no threshold, + min-one rule, global threshold, per-class thresholds tuned\n     on OOF (honest) vs tuned on the held-out set itself (leaky) and the resulting "leak gap"\n  3. a HEAD/TTA table: sigmoid / softmax / blend x with / without TTA (selected on OOF, reported on the held-out set)\n\nAll choices are made on OOF; the held-out set is only used to REPORT (except the deliberately leaky row, which is labelled).\n\nconfigs.json example:\n    {"light_aug": ["swin_tiny_s224"], "heavy_aug": ["swin_tiny_s224_heavy"], "fusion_gate": ["hybrid_gate_s224"]}\nA config with several prefixes is an equal-weight probability average of those runs.\n"""\nimport argparse\nimport json\nimport os\n\nimport numpy as np\nimport pandas as pd\n\nfrom . import evalkit as E\nfrom .run_ensemble import _load_run, _variants\n\n\ndef _config_probs(out_dir, prefixes, folds, pool_uids, n_classes, y_pool):\n    """OOF / holdout probabilities of a config: best head/TTA variant per run (chosen on OOF), equal-weight average."""\n    oofs, holds, per_fold_hold = [], [], []\n    chosen = {}\n    hold_uid = None\n    for p in prefixes:\n        d = _load_run(out_dir, p, folds, pool_uids, True, False, n_classes)\n        vs = _variants(d["oof"])\n        sc = {k: E.sample_mean_f1(y_pool, E.predict_labels(v, 0.5, True)) for k, v in vs.items()}\n        best = max(sc, key=sc.get)\n        chosen[p] = best\n        oofs.append(vs[best]); holds.append(_variants(d["hold"])[best]); hold_uid = d["hold_uid"]\n        # individual fold models on the holdout (spread across folds)\n        pf = []\n        for f in folds:\n            z = np.load(os.path.join(out_dir, f"{p}_f{f}", "preds.npz"))\n            key = {"sig": "sig", "soft": "soft", "sig_tta": "sig_tta", "soft_tta": "soft_tta"}.get(best)\n            h = z[f"hold_{key}"] if key else E.combine_heads(z["hold_sig" + ("_tta" if best.endswith("tta") else "")],\n                                                             z["hold_soft" + ("_tta" if best.endswith("tta") else "")], 0.5)\n            pf.append(h)\n        per_fold_hold.append(pf)\n    oof = E.average_probs(oofs)\n    hold = E.average_probs(holds)\n    fold_probs = [E.average_probs([per_fold_hold[r][i] for r in range(len(prefixes))]) for i in range(len(folds))]\n    return {"oof": oof, "hold": hold, "fold_hold": fold_probs, "chosen": chosen, "hold_uid": hold_uid}\n\n\ndef _md(df):\n    """Tiny markdown table writer (avoids the optional \'tabulate\' dependency of DataFrame.to_markdown)."""\n    cols = list(df.columns)\n    out = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]\n    for _, r in df.iterrows():\n        out.append("| " + " | ".join("" if (isinstance(v, float) and np.isnan(v)) else str(v) for v in r.tolist()) + " |")\n    return "\\n".join(out)\n\n\ndef _best_global_threshold(oof, y):\n    grid = E.DEFAULT_GRID\n    scores = [E.sample_mean_f1(y, E.predict_labels(oof, t, True)) for t in grid]\n    return float(grid[int(np.argmax(scores))])\n\n\ndef run(a):\n    os.makedirs(a.result_dir, exist_ok=True)\n    index = pd.read_csv(a.index_csv)\n    label_cols = [c for c in index.columns if c.startswith("label_")]\n    n = len(label_cols)\n    pool_uids = pd.read_csv(a.cv_folds_csv)["uid"].tolist()\n    lab = index.set_index("uid")\n    y_pool = lab.loc[pool_uids, label_cols].to_numpy(np.int64)\n    folds = [int(x) for x in a.folds.split(",")]\n    configs = json.load(open(a.config_json))\n    assert a.reference in configs, f"reference {a.reference} not in {list(configs)}"\n\n    cache, y_hold, groups = {}, None, None\n    for name, prefixes in configs.items():\n        cache[name] = _config_probs(a.out_dir, prefixes, folds, pool_uids, n, y_pool)\n        if y_hold is None:\n            order = cache[name]["hold_uid"].tolist()\n            y_hold = lab.loc[order, label_cols].to_numpy(np.int64)\n            groups = lab.loc[order, "patient_id"].astype(str).to_numpy()\n\n    ref_pred = E.predict_labels(cache[a.reference]["hold"], 0.5, True)\n    rows = []\n    for name, c in cache.items():\n        pred = E.predict_labels(c["hold"], 0.5, True)\n        ci = E.cluster_bootstrap(E.sample_mean_f1, y_hold, pred, groups, n_boot=a.n_boot)\n        fold_f1 = [E.sample_mean_f1(y_hold, E.predict_labels(p, 0.5, True)) for p in c["fold_hold"]]\n        row = {"config": name, "runs": "+".join(configs[name]),\n               "oof_sample_f1": E.sample_mean_f1(y_pool, E.predict_labels(c["oof"], 0.5, True)),\n               "hold_sample_f1": ci["estimate"], "hold_ci_low": ci["ci_low"], "hold_ci_high": ci["ci_high"],\n               "hold_macro_f1": E.macro_f1(y_hold, pred),\n               "oof_minus_hold": None,\n               "fold_models_mean": float(np.mean(fold_f1)), "fold_models_std": float(np.std(fold_f1, ddof=1)) if len(fold_f1) > 1 else 0.0}\n        row["oof_minus_hold"] = row["oof_sample_f1"] - row["hold_sample_f1"]      # large positive = validation over-optimistic\n        if name != a.reference:\n            d = E.paired_bootstrap(E.sample_mean_f1, y_hold, pred, ref_pred, groups, n_boot=a.n_boot)\n            row.update({"diff_vs_ref": d["diff"], "diff_ci_low": d["ci_low"], "diff_ci_high": d["ci_high"], "p_value": d["p_value"]})\n        rows.append(row)\n    cfg_df = pd.DataFrame(rows)\n    cfg_df.to_csv(os.path.join(a.result_dir, "ablation_configs.csv"), index=False)\n\n    # ---- decoding ablation on the reference config\n    ref = cache[a.reference]\n    oof, hold = ref["oof"], ref["hold"]\n    thr_oof = E.tune_thresholds_oof(oof, y_pool)\n    thr_leaky = E.tune_thresholds_oof(hold, y_hold)                 # DELIBERATELY leaky: tuned on what it is scored on\n    t_glob = _best_global_threshold(oof, y_pool)\n    rules = {"0.5, empty allowed": E.predict_labels(hold, 0.5, False),\n             "0.5 + min-one": E.predict_labels(hold, 0.5, True),\n             f"global threshold {t_glob:.2f} (tuned on OOF) + min-one": E.predict_labels(hold, t_glob, True),\n             "per-class thresholds tuned on OOF (honest) + min-one": E.predict_labels(hold, thr_oof, True),\n             "per-class thresholds tuned on HELD-OUT (leaky) + min-one": E.predict_labels(hold, thr_leaky, True)}\n    drows = []\n    for k, pred in rules.items():\n        ci = E.cluster_bootstrap(E.sample_mean_f1, y_hold, pred, groups, n_boot=a.n_boot)\n        drows.append({"decoding": k, "hold_sample_f1": ci["estimate"], "ci_low": ci["ci_low"], "ci_high": ci["ci_high"],\n                      "macro_f1": E.macro_f1(y_hold, pred)})\n    dec_df = pd.DataFrame(drows)\n    leak_gap = float(drows[4]["hold_sample_f1"] - drows[3]["hold_sample_f1"])\n    dec_df.to_csv(os.path.join(a.result_dir, "ablation_decoding.csv"), index=False)\n\n    # ---- head / TTA variants of the reference config\'s first run\n    p0 = configs[a.reference][0]\n    d0 = _load_run(a.out_dir, p0, folds, pool_uids, True, False, n)\n    vo, vh = _variants(d0["oof"]), _variants(d0["hold"])\n    hrows = [{"variant": k, "oof_sample_f1": E.sample_mean_f1(y_pool, E.predict_labels(vo[k], 0.5, True)),\n              "hold_sample_f1": E.sample_mean_f1(y_hold, E.predict_labels(vh[k], 0.5, True))} for k in vo]\n    head_df = pd.DataFrame(hrows)\n    head_df.to_csv(os.path.join(a.result_dir, "ablation_heads_tta.csv"), index=False)\n\n    md = ["# Ablation report", "", f"Reference: **{a.reference}**. Held-out set: {len(y_hold)} images, {len(set(groups))} patients; "\n          f"CI = patient bootstrap ({a.n_boot}).", "", "## Configurations", _md(cfg_df.round(4)), "",\n          "## Decoding (reference config)", _md(dec_df.round(4)), "",\n          f"Leak gap (leaky - honest per-class thresholds) = **{leak_gap:+.4f}**", "",\n          f"## Head / TTA variants of `{p0}`", _md(head_df.round(4)), ""]\n    open(os.path.join(a.result_dir, "ablation.md"), "w").write("\\n".join(md))\n    return {"configs": cfg_df, "decoding": dec_df, "heads": head_df, "leak_gap": leak_gap}\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--out_dir", required=True)\n    p.add_argument("--config_json", required=True)\n    p.add_argument("--reference", required=True)\n    p.add_argument("--index_csv", required=True)\n    p.add_argument("--cv_folds_csv", required=True)\n    p.add_argument("--result_dir", required=True)\n    p.add_argument("--folds", default="0,1,2,3,4")\n    p.add_argument("--n_boot", type=int, default=1000)\n    r = run(p.parse_args())\n    print(r["configs"].round(4).to_string(index=False)); print(r["decoding"].round(4).to_string(index=False))\n    print("leak gap:", round(r["leak_gap"], 4))\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote src/ablation_report.py")

In [ ]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/make_leaky_splits_v2.py", "w", encoding="utf-8") as f:
    f.write('"""\nmake_leaky_splits_v2 -- DELIBERATELY leaky image-level 5-fold split of the same CV pool (same held-out set), used only for\nthe split-type ablation ("how much does image-level splitting inflate the validation score?").\n\n    python -m src.make_leaky_splits_v2            # reads splits/cv_folds.csv, writes splits/cv_folds_leaky.csv\n"""\nimport os\n\nimport numpy as np\nimport pandas as pd\nfrom iterstrat.ml_stratifiers import MultilabelStratifiedKFold\n\nfrom . import config as C\n\n\ndef main():\n    src = os.path.join(C.INDEX_ROOT, "splits", "cv_folds.csv")\n    df = pd.read_csv(src)\n    label_cols = [c for c in df.columns if c.startswith("label_")]\n    Y = df[label_cols].to_numpy(int)\n    fold = np.full(len(df), -1)\n    for k, (_, val) in enumerate(MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=C.SEED).split(np.zeros(len(df)), Y)):\n        fold[val] = k\n    out = df.copy()\n    out["fold"] = fold\n    n_pat_multi = int((out.groupby("patient_id")["fold"].nunique() > 1).sum())\n    dst = os.path.join(C.INDEX_ROOT, "splits", "cv_folds_leaky.csv")\n    out.to_csv(dst, index=False)\n    print(f"image-level split: {n_pat_multi} / {out[\'patient_id\'].nunique()} patients have images in more than one fold (patient-level split: 0)")\n    print("saved", dst)\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote src/make_leaky_splits_v2.py")

In [ ]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/make_mura_official_splits.py", "w", encoding="utf-8") as f:
    f.write('"""\nmake_mura_official_splits -- MURA with the OFFICIAL protocol so results are comparable with published work (e.g. Shah et al. 2026,\nwho test on MURA\'s public validation set): held-out set = every image under MURA\'s  valid/  folder; CV pool = every image under\ntrain/, split into 5 PATIENT-level folds.  Replaces make_holdout_cv_splits for MURA.\n\n    python -m src.make_mura_official_splits        # needs index.csv from build_index_mura (uids start with "train__" / "valid__")\nWrites splits/holdout_uids.csv and splits/cv_folds.csv (same columns as the generic script).\n"""\nimport os\n\nimport numpy as np\nimport pandas as pd\nfrom iterstrat.ml_stratifiers import MultilabelStratifiedKFold\n\nfrom . import config as C\n\n\ndef split(index_df, seed=C.SEED, n_folds=5):\n    label_cols = [c for c in index_df.columns if c.startswith("label_")]\n    is_valid = index_df["uid"].str.startswith("valid__")\n    hold, pool = index_df[is_valid].reset_index(drop=True), index_df[~is_valid].reset_index(drop=True)\n    assert len(hold) and len(pool), "need both train__ and valid__ uids (is DATA_ROOT the MURA-v1.1 root?)"\n    overlap = set(hold["patient_id"]) & set(pool["patient_id"])\n    assert not overlap, f"{len(overlap)} patient ids appear in both train and valid"\n    pl = pool.groupby("patient_id")[label_cols].max().reset_index()\n    fold = np.full(len(pl), -1)\n    for k, (_, val) in enumerate(MultilabelStratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)\n                                 .split(np.zeros(len(pl)), pl[label_cols].to_numpy(int))):\n        fold[val] = k\n    pool["fold"] = pool["patient_id"].map(dict(zip(pl["patient_id"], fold)))\n    assert (pool["fold"] >= 0).all() and (pool.groupby("patient_id")["fold"].nunique() <= 1).all()\n    return hold[["uid", "patient_id"] + label_cols], pool[["uid", "patient_id", "fold"] + label_cols]\n\n\ndef main():\n    index = pd.read_csv(C.INDEX_CSV)\n    hold, pool = split(index)\n    out = os.path.join(C.INDEX_ROOT, "splits")\n    os.makedirs(out, exist_ok=True)\n    hold.to_csv(os.path.join(out, "holdout_uids.csv"), index=False)\n    pool.to_csv(os.path.join(out, "cv_folds.csv"), index=False)\n    print(f"held-out (MURA valid/): {len(hold)} images, {hold[\'patient_id\'].nunique()} patients | "\n          f"CV pool (train/): {len(pool)} images, {pool[\'patient_id\'].nunique()} patients")\n    print("per-fold images:", pool["fold"].value_counts().sort_index().to_dict())\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote src/make_mura_official_splits.py")

In [ ]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/build_index_mura.py", "w", encoding="utf-8") as f:
    f.write('"""\nStep 1a (MURA variant of build_index.py): builds index.csv for the Stanford\nMURA musculoskeletal radiograph dataset, re-tasked here as a BODY-PART\nclassification problem -- the same task UNIFESP solves -- instead of MURA\'s\nown normal/abnormal task. The body part each image shows is already\nencoded in its folder name (.../XR_ELBOW/patientNNNNN/study1_.../image1.png,\nofficial MURA v1.1 layout), so this is a relabeling, not new annotation.\n\nUnlike UNIFESP/ChestX-ray14, MURA is single-label here (each image shows\nexactly one of 7 body parts), not multilabel -- but a one-hot row is just a\ndegenerate multilabel row, so make_splits.py / make_holdout_cv_splits.py /\ntrain.py / train_hybrid.py / evaluate_on_holdout.py all run completely\nunchanged. "Same code, dataset path + last-layer neuron count (7) change."\n\nBoth MURA\'s train/ and valid/ folders are pooled here (every row marked\nsplit="train") since our own 80/20 holdout + 5-fold CV\n(make_holdout_cv_splits.py) replaces MURA\'s own split, same as for\nchestxray14 -- there\'s no Kaggle-leaderboard test set to preserve.\n\nPatient id is parsed from each image\'s path ("patientNNNNN"), for the same\npatient-level splitting make_splits.py already does (many patients have\nmultiple studies/images here too).\n\nOutput: <INDEX_ROOT>/index.csv with columns:\n    uid, split, patient_id, label_0_Elbow, ...\n\nKaggle usage (attach cjinny/mura-v11 as the images dataset, then):\n    import os\n    os.environ["UNIFESP_DATASET"] = "mura"\n    os.environ["UNIFESP_DATA_ROOT"] = "/kaggle/input/mura-v11"\n    os.environ["UNIFESP_INDEX_ROOT"] = "/kaggle/working"\n    os.environ["UNIFESP_OUTPUT_DIR"] = "/kaggle/working/runs"\n    !python -m src.build_index_mura\n"""\nimport os\nimport re\n\nimport pandas as pd\n\nfrom . import config as C\nfrom .dataset import build_uid_to_path\n\nLABEL_NAMES = C.LABEL_NAMES\nN_CLASSES = C.N_CLASSES\n\nBODY_PART_RE = re.compile(r"XR_([A-Z]+)")\nPATIENT_RE = re.compile(r"(patient\\d+)", re.IGNORECASE)\n\n\ndef parse_body_part(path):\n    m = BODY_PART_RE.search(path.upper())\n    return m.group(1).title() if m else None\n\n\ndef parse_patient_id(path):\n    m = PATIENT_RE.search(path)\n    return m.group(1) if m else None\n\n\ndef main():\n    assert C.DATASET == "mura", (\n        f"UNIFESP_DATASET={C.DATASET!r}, expected \'mura\' -- set the env var "\n        f"before running this (config.py picks the 7-class LABEL_NAMES from it)."\n    )\n\n    print("Indexing PNG files under", C.DATA_ROOT, "...")\n    uid_to_path = build_uid_to_path(C.DATA_ROOT)\n    print(f"found {len(uid_to_path)} .png files")\n\n    name_to_label_idx = {name: k for k, name in LABEL_NAMES.items()}\n\n    rows = []\n    skipped = 0\n    for uid, path in uid_to_path.items():\n        body_part = parse_body_part(path)\n        patient_id = parse_patient_id(path)\n        if body_part is None or patient_id is None or body_part not in name_to_label_idx:\n            skipped += 1\n            continue\n        rows.append({"uid": uid, "split": "train", "patient_id": patient_id, "body_part": body_part})\n    if skipped:\n        print(f"WARN: {skipped} file(s) didn\'t match the expected "\n              f".../XR_<PART>/patientNNNNN/... path pattern, skipped")\n\n    index_df = pd.DataFrame(rows)\n\n    for k in range(N_CLASSES):\n        name = LABEL_NAMES[k]\n        col = f"label_{k}_{name.replace(\' \', \'_\')}"\n        index_df[col] = (index_df["body_part"] == name).astype(int)\n\n    index_df = index_df.drop(columns=["body_part"])\n\n    os.makedirs(C.INDEX_ROOT, exist_ok=True)\n    out_path = os.path.join(C.INDEX_ROOT, "index.csv")\n    index_df.to_csv(out_path, index=False)\n    print("Saved index to", out_path, "| rows:", len(index_df))\n\n    # ---- sanity checks (mirrors build_index.py\'s patient-level checks) ----\n    n_patients = index_df["patient_id"].nunique()\n    dup = index_df.groupby("patient_id").size()\n    multi_image_patients = (dup > 1).sum()\n    print(f"\\nUnique patients: {n_patients}")\n    print(f"Patients with >1 image: {multi_image_patients} / {n_patients}")\n\n    label_cols = [c for c in index_df.columns if c.startswith("label_")]\n    print("\\nPer-class image count:\\n", index_df[label_cols].sum())\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote src/build_index_mura.py")

In [ ]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/build_index_covidcxr.py", "w", encoding="utf-8") as f:
    f.write('"""\nStep 1a (COVID-19 Radiography Database variant of build_index.py): builds\nindex.csv for the Qatar University / University of Dhaka COVID-19\nRadiography Database, re-tasked as 4-class chest X-ray diagnosis\nclassification (COVID / Lung_Opacity / Normal / Viral_Pneumonia) -- the\nclass is already encoded in each image\'s top-level folder name\n(official Kaggle layout: COVID-19_Radiography_Dataset/<Class>/images/*.png,\nplus a parallel <Class>/masks/*.png tree of lung-segmentation masks that\nbuild_uid_to_path (dataset.py) already skips).\n\nUnlike UNIFESP/ChestX-ray14 this is single-label (one diagnosis per image),\nsame as MURA -- a one-hot row is just a degenerate multilabel row, so\nevery downstream script runs unchanged. "Same code, dataset path +\nlast-layer neuron count (4) change."\n\nNo patient-level grouping here: this dataset is aggregated from many\ndifferent source datasets/publications (SIRM, Cohen\'s GitHub repo, the\nKaggle pneumonia dataset, 43 publications, ...) with no single consistent\npatient-id field across all of them, so each image\'s own uid is used as\nits own "patient_id" below -- i.e. an ordinary image-level split, not\npatient-level. Worth naming as a limitation in the paper (the same reason\npatient-level splitting matters for the other datasets applies here too,\nwe just don\'t have the id to do it).\n\nOutput: <INDEX_ROOT>/index.csv with columns:\n    uid, split, patient_id, label_0_COVID, ...\n\nKaggle usage (attach tawsifurrahman/covid19-radiography-database, then):\n    import os\n    os.environ["UNIFESP_DATASET"] = "covidcxr"\n    os.environ["UNIFESP_DATA_ROOT"] = "/kaggle/input/covid19-radiography-database"\n    os.environ["UNIFESP_INDEX_ROOT"] = "/kaggle/working"\n    os.environ["UNIFESP_OUTPUT_DIR"] = "/kaggle/working/runs"\n    !python -m src.build_index_covidcxr\n"""\nimport os\n\nimport pandas as pd\n\nfrom . import config as C\nfrom .dataset import build_uid_to_path\n\nLABEL_NAMES = C.LABEL_NAMES\nN_CLASSES = C.N_CLASSES\n\n\ndef parse_class_name(path, name_to_label_idx):\n    # path components are like .../COVID-19_Radiography_Dataset/COVID/images/COVID-1.png\n    # (or "Viral Pneumonia" with a space, or "Lung_Opacity") -- match against\n    # the known class names directly rather than assuming exact folder casing.\n    parts = [p.replace(" ", "_") for p in os.path.normpath(path).split(os.sep)]\n    for p in parts:\n        for name in name_to_label_idx:\n            if p.lower() == name.lower():\n                return name\n    return None\n\n\ndef main():\n    assert C.DATASET == "covidcxr", (\n        f"UNIFESP_DATASET={C.DATASET!r}, expected \'covidcxr\' -- set the env var "\n        f"before running this (config.py picks the 4-class LABEL_NAMES from it)."\n    )\n\n    print("Indexing PNG files under", C.DATA_ROOT, "...")\n    uid_to_path = build_uid_to_path(C.DATA_ROOT)\n    print(f"found {len(uid_to_path)} .png files (mask files already excluded)")\n\n    name_to_label_idx = {name: k for k, name in LABEL_NAMES.items()}\n\n    rows = []\n    skipped = 0\n    for uid, path in uid_to_path.items():\n        class_name = parse_class_name(path, name_to_label_idx)\n        if class_name is None:\n            skipped += 1\n            continue\n        # no shared patient id across this dataset\'s many source collections\n        # -- fall back to image-level "patients" (see module docstring)\n        rows.append({"uid": uid, "split": "train", "patient_id": uid, "class_name": class_name})\n    if skipped:\n        print(f"WARN: {skipped} file(s) didn\'t match any of {list(name_to_label_idx)}, skipped")\n\n    index_df = pd.DataFrame(rows)\n\n    for k in range(N_CLASSES):\n        name = LABEL_NAMES[k]\n        col = f"label_{k}_{name.replace(\' \', \'_\')}"\n        index_df[col] = (index_df["class_name"] == name).astype(int)\n\n    index_df = index_df.drop(columns=["class_name"])\n\n    os.makedirs(C.INDEX_ROOT, exist_ok=True)\n    out_path = os.path.join(C.INDEX_ROOT, "index.csv")\n    index_df.to_csv(out_path, index=False)\n    print("Saved index to", out_path, "| rows:", len(index_df))\n\n    label_cols = [c for c in index_df.columns if c.startswith("label_")]\n    print("\\nPer-class image count:\\n", index_df[label_cols].sum())\n    print(f"\\nNote: patient_id == uid for every row (no shared patient metadata "\n          f"across this dataset\'s source collections) -- splits are image-level, not patient-level.")\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote src/build_index_covidcxr.py")

In [ ]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/build_index_chestxray14.py", "w", encoding="utf-8") as f:
    f.write('"""\nStep 1a (ChestX-ray14 variant of build_index.py): builds index.csv for the\nNIH ChestX-ray14 dataset in the exact schema build_index.py produces for\nUNIFESP, so make_splits.py / make_holdout_cv_splits.py / train.py /\ntrain_hybrid.py / evaluate_on_holdout.py all run completely unchanged --\n"same code, just dataset path + last-layer neuron count change" (the neuron\ncount changes automatically: see config.py\'s DATASET switch).\n\nReads Data_entry_2017.csv\'s "Finding Labels" column (pipe-separated, e.g.\n"Cardiomegaly|Effusion", or "No Finding") into one-hot label_k_<Name>\ncolumns. "No Finding" is deliberately not its own class -- it\'s the\nall-zero label vector, the same multilabel convention the CheXNet-style\nliterature on this dataset uses, and engine.py\'s sample_mean_f1/\ncompute_metrics already handle an all-zero row correctly.\n\nEvery row is marked split="train" -- there\'s no Kaggle-leaderboard test set\nto preserve here. The pipeline\'s own 80/20 holdout is carved out downstream\nby make_holdout_cv_splits.py, same as for UNIFESP.\n\nOutput: <INDEX_ROOT>/index.csv with columns:\n    uid, split, patient_id, label_0_Atelectasis, ...\n\nKaggle usage (attach nih-chest-xrays/data as the images dataset, then):\n    import os\n    os.environ["UNIFESP_DATASET"] = "chestxray14"\n    os.environ["UNIFESP_DATA_ROOT"] = "/kaggle/input/data"\n    os.environ["UNIFESP_INDEX_ROOT"] = "/kaggle/working"\n    os.environ["UNIFESP_OUTPUT_DIR"] = "/kaggle/working/runs"\n    !python -m src.build_index_chestxray14\n"""\nimport os\n\nimport pandas as pd\n\nfrom . import config as C\nfrom .dataset import build_uid_to_path\n\nLABEL_NAMES = C.LABEL_NAMES\nN_CLASSES = C.N_CLASSES\n\n\nENTRY_CSV_NAMES = {"data_entry_2017.csv", "sample_labels.csv"}  # full dataset vs. the 5% nih-chest-xrays/sample\n\n\ndef find_entry_csv(data_root):\n    for root, _dirs, files in os.walk(data_root):\n        for f in files:\n            if f.lower() in ENTRY_CSV_NAMES:\n                return os.path.join(root, f)\n    raise FileNotFoundError(\n        f"Neither Data_entry_2017.csv nor sample_labels.csv found anywhere under {data_root}. "\n        f"Check UNIFESP_DATA_ROOT points at the attached nih-chest-xrays/data (or nih-chest-xrays/sample) dataset."\n    )\n\n\ndef main():\n    assert C.DATASET == "chestxray14", (\n        f"UNIFESP_DATASET={C.DATASET!r}, expected \'chestxray14\' -- set the env var "\n        f"before running this (config.py picks the 14-class LABEL_NAMES from it)."\n    )\n\n    print("Indexing PNG files under", C.DATA_ROOT, "...")\n    uid_to_path = build_uid_to_path(C.DATA_ROOT)\n    print(f"found {len(uid_to_path)} .png files")\n\n    entry_csv = find_entry_csv(C.DATA_ROOT)\n    entry_df = pd.read_csv(entry_csv)\n    entry_df["uid"] = entry_df["Image Index"].str.replace(".png", "", regex=False)\n    meta_by_uid = entry_df.set_index("uid")[["Patient ID", "Finding Labels"]].to_dict("index")\n\n    rows = []\n    missing_meta = 0\n    for uid in uid_to_path:\n        meta = meta_by_uid.get(uid)\n        if meta is None:\n            missing_meta += 1\n            continue\n        rows.append({\n            "uid": uid,\n            "split": "train",\n            "patient_id": str(meta["Patient ID"]),\n            "finding_labels": meta["Finding Labels"],\n        })\n    if missing_meta:\n        print(f"WARN: {missing_meta} PNG file(s) had no matching row in Data_entry_2017.csv, skipped")\n\n    index_df = pd.DataFrame(rows)\n\n    for k in range(N_CLASSES):\n        name = LABEL_NAMES[k]\n        col = f"label_{k}_{name.replace(\' \', \'_\')}"\n        index_df[col] = index_df["finding_labels"].apply(\n            lambda s, name=name: int(name in str(s).split("|"))\n        )\n\n    index_df = index_df.drop(columns=["finding_labels"])\n\n    os.makedirs(C.INDEX_ROOT, exist_ok=True)\n    out_path = os.path.join(C.INDEX_ROOT, "index.csv")\n    index_df.to_csv(out_path, index=False)\n    print("Saved index to", out_path, "| rows:", len(index_df))\n\n    # ---- sanity checks (mirrors build_index.py\'s patient-level checks) ----\n    n_patients = index_df["patient_id"].nunique()\n    dup = index_df.groupby("patient_id").size()\n    multi_image_patients = (dup > 1).sum()\n    print(f"\\nUnique patients: {n_patients}")\n    print(f"Patients with >1 image: {multi_image_patients} / {n_patients}")\n\n    label_cols = [c for c in index_df.columns if c.startswith("label_")]\n    print("\\nPer-class positive image count:\\n", index_df[label_cols].sum())\n    no_finding = (index_df[label_cols].sum(axis=1) == 0).sum()\n    print(f"\\n\'No Finding\' images (all-zero label vector): {no_finding} / {len(index_df)}")\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote src/build_index_chestxray14.py")

In [ ]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/make_pseudo_labels.py", "w", encoding="utf-8") as f:
    f.write('"""\nmake_pseudo_labels -- confident pseudo-labels for the UNLABELLED Kaggle test images, from a finished ensemble.\n\nThe competition rules (read on the join dialog) say: "Annotating the test set is manually prohibited (pseudo-label is allowed)".\nThis script does NOT touch any true test label (there are none); it only turns the ensemble\'s own predictions into training targets.\n\n    python -m src.make_pseudo_labels --ensemble_arrays results_s2a/ensemble_arrays.npz --index_csv index.csv --out splits/pseudo.csv --min_conf 0.9\n\nSelection (per image): decode with the ensemble\'s frozen thresholds (min-one rule); keep the image only if EVERY predicted class has probability >= min_conf\nand EVERY other class has probability <= 1 - min_conf.  Optional --max_images caps the number kept (most confident first).\nOutput columns: uid + label_* (hard 0/1), compatible with train_v2 --pseudo_csv.\n\nHonest-evaluation note: pseudo-labels come from models that saw all training folds, so the OOF score of a pseudo-label round is slightly optimistic.\nJudge such a round by the Late-Submission score, and report that it is a transductive (test-image-using) variant.\n"""\nimport argparse\n\nimport numpy as np\nimport pandas as pd\n\nfrom . import evalkit as E\n\n\ndef select_confident(prob, thresholds, min_conf=0.9, max_images=0):\n    pred = E.predict_labels(prob, thresholds, min_one=True)\n    pos_ok = np.where(pred == 1, prob >= min_conf, True).all(1)\n    neg_ok = np.where(pred == 0, prob <= 1.0 - min_conf, True).all(1)\n    keep = pos_ok & neg_ok\n    conf = np.where(pred == 1, prob, 1.0 - prob).min(1)\n    idx = np.where(keep)[0]\n    if max_images and len(idx) > max_images:\n        idx = idx[np.argsort(-conf[idx])[:max_images]]\n    return np.sort(idx), pred\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--ensemble_arrays", required=True)\n    p.add_argument("--index_csv", required=True)\n    p.add_argument("--out", required=True)\n    p.add_argument("--min_conf", type=float, default=0.9)\n    p.add_argument("--max_images", type=int, default=0)\n    a = p.parse_args()\n    z = np.load(a.ensemble_arrays, allow_pickle=False)\n    assert z["test"].size, "ensemble_arrays.npz has no test probabilities"\n    cols = [c for c in pd.read_csv(a.index_csv, nrows=1).columns if c.startswith("label_")]\n    assert z["test"].shape[1] == len(cols), (z["test"].shape, len(cols))\n    idx, pred = select_confident(z["test"], z["thresholds"], a.min_conf, a.max_images)\n    out = pd.DataFrame(pred[idx], columns=cols)\n    out.insert(0, "uid", z["test_uid"][idx])\n    out.to_csv(a.out, index=False)\n    n = len(z["test"])\n    print(f"kept {len(idx)} / {n} test images (min_conf={a.min_conf}); "\n          f"single-label share {float((pred[idx].sum(1) == 1).mean()):.3f}; class counts {dict(zip(cols, pred[idx].sum(0).tolist()))}")\n    print("saved", a.out)\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote src/make_pseudo_labels.py")

In [ ]:
import os
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/explain.py", "w", encoding="utf-8") as f:
    f.write('"""\nexplain -- explainability for the body-part models: Grad-CAM + a quantitative check that the maps mean something + error analysis.\n\nWhat it produces for ONE trained model (a fold model\'s final_ema.pt) on the held-out images:\n  1. gallery_page*.png   per class: a correctly predicted image and a missed image, with Grad-CAM for the true class and for the wrongly predicted class\n  2. explain_metrics.json\n       * faithfulness (deletion test): delete the most-salient pixels first and watch the class probability fall; compared with deleting\n         in random order.  ``deletion_gain`` = (random score - CAM score) > 0 means the map points at pixels the model really uses.\n       * sanity check (Adebayo et al. 2018 style): CAM of the trained model vs CAM of a randomly initialised model of the same architecture\n         (Spearman rank correlation); a map that does not change when the weights are randomised is an edge detector, not an explanation.\n  3. top_confusions.csv  most frequent (true class missed -> class predicted instead) pairs  -> error analysis table\nWorks for CNNs (NCHW feature maps) and Swin (NHWC feature maps) without a per-architecture table: the target layer is found by a probe forward\npass (the last module that outputs a spatial 4-D tensor).\n\nLIMITATIONS to state in the paper: Grad-CAM maps are coarse (7x7 to 8x8 feature grid), there are NO ground-truth localisation labels in UNIFESP (so no\npointing-game / IoU), and no radiologist has rated the maps unless you add that review.\n\n    python -m src.explain --weights runs/convnext_tiny_s256_f0/final_ema.pt --backbone convnext_tiny --img_size 256 \\\n        --index_csv index.csv --holdout_csv splits/holdout_uids.csv --out_dir explain_convnext --title convnext_tiny\n"""\nimport argparse\nimport json\nimport os\n\nimport numpy as np\n\nfrom . import evalkit as E\n\nMEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)\n\n\n# ------------------------------------------------------------------------------------------------ Grad-CAM\ndef is_nhwc(s):\n    """Shape (B, a, b, c): channels-last if the two equal dims are 1,2 (square spatial first), channels-first if they are 2,3; else the\n    larger of dim 1 / dim 3 is the channel dim."""\n    if s[2] == s[3] and s[1] != s[2]:\n        return False\n    if s[1] == s[2] and s[3] != s[2]:\n        return True\n    return s[-1] > s[1]\n\n\ndef auto_target(model, img_size, device="cpu"):\n    """(module_name, channels_last) of the last module whose output is a spatial 4-D feature map (>=2x2)."""\n    import torch\n    found, hooks = [], []\n    for name, m in model.named_modules():\n        if name:\n            def mk(n):\n                def f(_m, _i, o):\n                    if torch.is_tensor(o) and o.ndim == 4:\n                        found.append((n, tuple(o.shape)))\n                return f\n            hooks.append(m.register_forward_hook(mk(name)))\n    with torch.no_grad():\n        model.eval()(torch.zeros(1, 3, img_size, img_size, device=device))\n    for h in hooks:\n        h.remove()\n    best = None\n    for n, s in found:\n        cl = is_nhwc(s)\n        hw = (s[1], s[2]) if cl else (s[2], s[3])\n        if min(hw) >= 2:\n            best = (n, cl)\n    assert best is not None, "no spatial feature map found"\n    return best\n\n\nclass GradCAM:\n    def __init__(self, model, img_size=224, device="cpu"):\n        self.model = model.eval()\n        self.name, self.channels_last = auto_target(model, img_size, device)\n        self._act = None\n        self._h = dict(model.named_modules())[self.name].register_forward_hook(lambda m, i, o: setattr(self, "_act", o))\n\n    def close(self):\n        self._h.remove()\n\n    def __call__(self, x, class_idx, head="sig"):\n        """x: (B,3,S,S) normalised tensor; class_idx: (B,) ints. Returns cam (B,h,w) in [0,1], sigmoid probs, softmax probs."""\n        import torch\n        with torch.enable_grad():\n            sig, soft = self.model(x)\n            out = sig if head == "sig" else soft\n            idx = torch.as_tensor(class_idx, device=out.device).long()\n            score = out[torch.arange(len(idx)), idx].sum()\n            act = self._act\n            grad = torch.autograd.grad(score, act)[0]\n        if self.channels_last:\n            act, grad = act.permute(0, 3, 1, 2), grad.permute(0, 3, 1, 2)\n        cam = torch.relu((grad.mean(dim=(2, 3), keepdim=True) * act).sum(1))\n        flat = cam.flatten(1)\n        mn, mx = flat.min(1)[0][:, None, None], flat.max(1)[0][:, None, None]\n        cam = (cam - mn) / (mx - mn + 1e-8)\n        return cam.detach().cpu().numpy(), torch.sigmoid(sig).detach().cpu().numpy(), torch.softmax(soft, 1).detach().cpu().numpy()\n\n\ndef upsample(cam, size):\n    import torch\n    t = torch.as_tensor(cam, dtype=torch.float32)[None, None]\n    return torch.nn.functional.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)[0, 0].numpy()\n\n\n# ------------------------------------------------------------------------------------------------ faithfulness / sanity\ndef trapezoid(y, x):\n    y, x = np.asarray(y, float), np.asarray(x, float)\n    return float(((y[:-1] + y[1:]) / 2 * np.diff(x)).sum())\n\n\ndef deletion_curve(model, x, order, class_idx, steps):\n    """Replace the first k pixels of ``order`` (flattened pixel ranking) with the per-channel image mean; return P(class) at each step."""\n    import torch\n    S = x.shape[-1]\n    fill = x.mean(dim=(2, 3), keepdim=True)\n    probs = []\n    for f in steps:\n        k = int(f * S * S)\n        m = torch.zeros(S * S, device=x.device)\n        m[torch.as_tensor(order[:k], device=x.device)] = 1.0\n        m = m.view(1, 1, S, S)\n        with torch.no_grad():\n            probs.append(float(torch.sigmoid(model(x * (1 - m) + fill * m)[0])[0, class_idx]))\n    return np.array(probs)\n\n\ndef deletion_scores(model, x, cam_up, class_idx, steps=(0, .05, .1, .15, .2, .3, .4, .5), n_random=3, seed=0, grid=None):\n    """Mean P(class) over the deletion range for CAM order vs random order (lower = deleting hurt more).\n    ``grid`` = (h, w) of the CAM\'s native feature grid: the random baseline is a random map on that SAME coarse grid, bilinearly upsampled like the CAM,\n    so it deletes blocks of the same granularity (a pixel-scatter baseline is far more destructive than a block and would make any CAM look bad)."""\n    rng = np.random.default_rng(seed)\n    S = cam_up.shape[0]\n    cam_p = deletion_curve(model, x, np.argsort(-cam_up.ravel()), class_idx, steps)\n    if grid is None:\n        rnd_orders = [rng.permutation(cam_up.size) for _ in range(n_random)]\n    else:\n        rnd_orders = [np.argsort(-upsample(rng.random(grid), S).ravel()) for _ in range(n_random)]\n    rnd = [deletion_curve(model, x, o, class_idx, steps) for o in rnd_orders]\n    s_cam = trapezoid(cam_p, steps) / steps[-1]\n    s_rnd = float(np.mean([trapezoid(r, steps) / steps[-1] for r in rnd]))\n    return s_cam, s_rnd, cam_p[0]\n\n\ndef spearman(a, b):\n    """Spearman rank correlation with AVERAGE ranks for ties (CAMs contain many exact zeros after ReLU / upsampling;\n    a plain argsort-of-argsort ranks ties by position and would invent a correlation)."""\n    from scipy.stats import rankdata\n    a, b = np.asarray(a, float).ravel(), np.asarray(b, float).ravel()\n    ra, rb = rankdata(a, method="average"), rankdata(b, method="average")\n    if ra.std() == 0 or rb.std() == 0:\n        return 0.0\n    return float(np.corrcoef(ra, rb)[0, 1])\n\n\ndef boot_ci(v, n=2000, seed=0):\n    v = np.asarray(v, float)\n    rng = np.random.default_rng(seed)\n    m = np.array([v[rng.integers(0, len(v), len(v))].mean() for _ in range(n)])\n    return {"mean": float(v.mean()), "ci_low": float(np.quantile(m, .025)), "ci_high": float(np.quantile(m, .975)), "n": int(len(v))}\n\n\n# ------------------------------------------------------------------------------------------------ driver\ndef run(a):\n    import torch\n    import pandas as pd\n    from . import config as C\n    from .dataset import UnifespDataset, build_uid_to_path\n    from .train_v2 import build_model, get_transforms\n    device = torch.device("cuda" if torch.cuda.is_available() and not a.cpu else "cpu")\n    names = [C.LABEL_NAMES[k] for k in sorted(C.LABEL_NAMES)]\n    N = len(names)\n    label_cols = [f"label_{k}_{C.LABEL_NAMES[k].replace(\' \', \'_\')}" for k in range(N)]\n    os.makedirs(a.out_dir, exist_ok=True)\n\n    model = build_model(a.backbone, pretrained=False)\n    model.load_state_dict(torch.load(a.weights, map_location="cpu", weights_only=True))\n    model.to(device).eval()\n    index = pd.read_csv(a.index_csv)\n    hold = index[index["uid"].isin(set(pd.read_csv(a.holdout_csv)["uid"]))].reset_index(drop=True)\n    if a.max_images:\n        hold = hold.iloc[:a.max_images]\n    ds = UnifespDataset(hold, C.DATA_ROOT, transform=get_transforms(a.img_size, False), uid_to_path=build_uid_to_path(C.DATA_ROOT))\n    X = torch.stack([ds[i][0] for i in range(len(ds))])\n    Y = hold[label_cols].to_numpy(np.int64)\n\n    sig, soft = [], []\n    with torch.no_grad():\n        for i in range(0, len(X), 32):\n            s, f = model(X[i:i + 32].to(device))\n            sig.append(torch.sigmoid(s.float()).cpu().numpy()); soft.append(torch.softmax(f.float(), 1).cpu().numpy())\n    blend = E.combine_heads(np.concatenate(sig), np.concatenate(soft), 0.5)\n    pred = E.predict_labels(blend, 0.5, min_one=True)\n    metrics = {"title": a.title or a.backbone, "weights": a.weights, "n_images": int(len(X)), "holdout_sample_f1_blend": E.sample_mean_f1(Y, pred)}\n\n    cam_engine = GradCAM(model, a.img_size, device)\n    metrics["target_layer"] = cam_engine.name\n    metrics["channels_last"] = bool(cam_engine.channels_last)\n    mean_t, std_t = torch.tensor(MEAN).view(1, 3, 1, 1), torch.tensor(STD).view(1, 3, 1, 1)\n\n    def cam_for(i, cls):\n        x = X[i:i + 1].to(device)\n        cam, _, _ = cam_engine(x, [cls])\n        return upsample(cam[0], a.img_size), cam[0].shape\n\n    def disp(i):\n        return (X[i:i + 1] * std_t + mean_t).clamp(0, 1)[0].permute(1, 2, 0).numpy()\n\n    # ---- gallery: per class one correct and one missed example\n    rows = []\n    for c in range(N):\n        pos = np.where(Y[:, c] == 1)[0]\n        ok = [i for i in pos if pred[i, c] == 1]\n        bad = [i for i in pos if pred[i, c] == 0]\n        if ok:\n            i = max(ok, key=lambda j: blend[j, c]); rows.append((c, "correct", i, c, c))\n        if bad:\n            i = min(bad, key=lambda j: blend[j, c]); wrong = int(np.argmax(np.where(Y[i] == 1, -1, blend[i]))); rows.append((c, "missed", i, c, wrong))\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    per_page = 6\n    for pg in range(0, len(rows), per_page):\n        chunk = rows[pg:pg + per_page]\n        fig, ax = plt.subplots(len(chunk), 3, figsize=(7.5, 2.5 * len(chunk)), squeeze=False)\n        for r, (c, kind, i, tc, wc) in enumerate(chunk):\n            img = disp(i)\n            ax[r, 0].imshow(img); ax[r, 0].set_title(f"{kind}: true {names[c]}", fontsize=8)\n            for col, cls, lab in ((1, tc, "true"), (2, wc, "predicted")):\n                ax[r, col].imshow(img); ax[r, col].imshow(cam_for(i, cls)[0], cmap="jet", alpha=0.45)\n                ax[r, col].set_title(f"CAM {lab}: {names[cls]} (p={blend[i, cls]:.2f})", fontsize=8)\n            for k in range(3):\n                ax[r, k].axis("off")\n        fig.tight_layout(); fig.savefig(os.path.join(a.out_dir, f"gallery_page{pg // per_page + 1}.png"), dpi=130); plt.close(fig)\n    metrics["gallery_rows"] = len(rows)\n\n    # ---- faithfulness (deletion) and randomisation sanity check on predicted-correct (image, class) pairs\n    pairs = [(i, int(c)) for i in range(len(X)) for c in np.where((Y[i] == 1) & (pred[i] == 1))[0]]\n    rng = np.random.default_rng(a.seed)\n    if len(pairs) > a.max_pairs:\n        pairs = [pairs[j] for j in rng.choice(len(pairs), a.max_pairs, replace=False)]\n    torch.manual_seed(a.seed)\n    rnd_model = build_model(a.backbone, pretrained=False).to(device).eval()\n    rnd_engine = GradCAM(rnd_model, a.img_size, device)\n    d_cam, d_rnd, rho = [], [], []\n    for i, c in pairs:\n        up, grid = cam_for(i, c)\n        sc, sr, _ = deletion_scores(model, X[i:i + 1].to(device), up, c, seed=a.seed + i, grid=grid)\n        d_cam.append(sc); d_rnd.append(sr)\n        cam_r, _, _ = rnd_engine(X[i:i + 1].to(device), [c])\n        rho.append(spearman(up, upsample(cam_r[0], a.img_size)))\n    d_cam, d_rnd, rho = np.array(d_cam), np.array(d_rnd), np.array(rho)\n    metrics["faithfulness_deletion"] = {"pairs": int(len(pairs)), "score_cam": boot_ci(d_cam), "score_random_order": boot_ci(d_rnd),\n                                        "deletion_gain_random_minus_cam": boot_ci(d_rnd - d_cam),\n                                        "note": "mean P(class) over deleting 0-50 % of pixels; lower = more faithful; gain > 0 means CAM beats a random map of the same grid granularity"}\n    metrics["randomization_sanity"] = {"spearman_trained_vs_random_weights": boot_ci(rho), "fraction_abs_rho_below_0.3": float((np.abs(rho) < 0.3).mean()),\n                                       "note": "low correlation = the map depends on the learned weights (passes the check)"}\n\n    # ---- error analysis\n    conf = {}\n    for i in range(len(Y)):\n        for t in np.where((Y[i] == 1) & (pred[i] == 0))[0]:\n            for p in np.where((Y[i] == 0) & (pred[i] == 1))[0]:\n                conf[(names[t], names[p])] = conf.get((names[t], names[p]), 0) + 1\n    tc = pd.DataFrame([{"true_class_missed": t, "predicted_instead": p, "count": n} for (t, p), n in conf.items()]).sort_values("count", ascending=False) if conf else \\\n        pd.DataFrame(columns=["true_class_missed", "predicted_instead", "count"])\n    tc.to_csv(os.path.join(a.out_dir, "top_confusions.csv"), index=False)\n    per_class = pd.DataFrame({"class": names, "n_true": Y.sum(0), "recall": [float(((pred[:, c] == 1) & (Y[:, c] == 1)).sum() / max(1, Y[:, c].sum())) for c in range(N)]})\n    per_class.to_csv(os.path.join(a.out_dir, "per_class_recall.csv"), index=False)\n    json.dump(metrics, open(os.path.join(a.out_dir, "explain_metrics.json"), "w"), indent=1)\n    print(json.dumps({k: metrics[k] for k in ("title", "n_images", "target_layer", "faithfulness_deletion", "randomization_sanity")}, indent=1)[:2500])\n    return metrics\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--weights", required=True)\n    p.add_argument("--backbone", required=True)\n    p.add_argument("--img_size", type=int, default=224)\n    p.add_argument("--index_csv", required=True)\n    p.add_argument("--holdout_csv", required=True)\n    p.add_argument("--out_dir", required=True)\n    p.add_argument("--title", default=None)\n    p.add_argument("--max_images", type=int, default=0)\n    p.add_argument("--max_pairs", type=int, default=150)\n    p.add_argument("--seed", type=int, default=0)\n    p.add_argument("--cpu", action="store_true")\n    run(p.parse_args())\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote src/explain.py")

## Environment, index, ALL-data folds, cache

In [ ]:
import sys, os
sys.path.insert(0, "/kaggle/working")
os.environ["UNIFESP_DATA_ROOT"] = IMAGES_DATASET
os.environ["UNIFESP_INDEX_ROOT"] = "/kaggle/working"
os.environ["UNIFESP_OUTPUT_DIR"] = "/kaggle/working/runs"
os.environ["UNIFESP_CACHE_DIR"] = "/kaggle/working/image_cache"
!pip install -q timm albumentations iterative-stratification pydicom
import torch
print("CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count(), "| torch", torch.__version__)

In [ ]:
!cd /kaggle/working && python -m src.build_index
!cd /kaggle/working && python -m src.make_splits
!cd /kaggle/working && python -m src.merge_runs --write_hashes --index_csv index.csv --splits_dir splits --out split_hashes.json

In [ ]:
!cd /kaggle/working && python -m src.cache_images

## 2a -- all-data folds, no pseudo-labels

In [ ]:
import subprocess, itertools, threading, queue, os, time

EPOCHS, BATCH = 30, 32
RUNS = [
 {
  "prefix": "s2_convnext_tiny_s256_seed0",
  "bb": "convnext_tiny",
  "size": 256,
  "lr": 0.0001,
  "aug": "light",
  "seed": 0,
  "folds_csv": "/kaggle/working/splits/folds.csv"
 },
 {
  "prefix": "s2_convnext_tiny_s256_seed1",
  "bb": "convnext_tiny",
  "size": 256,
  "lr": 0.0001,
  "aug": "light",
  "seed": 1,
  "folds_csv": "/kaggle/working/splits/folds.csv"
 },
 {
  "prefix": "s2_resnet50_s256_heavy_seed0",
  "bb": "resnet50",
  "size": 256,
  "lr": 0.0003,
  "aug": "heavy",
  "seed": 0,
  "folds_csv": "/kaggle/working/splits/folds.csv"
 },
 {
  "prefix": "s2_resnet50_s256_heavy_seed1",
  "bb": "resnet50",
  "size": 256,
  "lr": 0.0003,
  "aug": "heavy",
  "seed": 1,
  "folds_csv": "/kaggle/working/splits/folds.csv"
 },
 {
  "prefix": "s2_efficientnet_b3_s256_hv_seed0",
  "bb": "efficientnet_b3",
  "size": 256,
  "lr": 0.0003,
  "aug": "heavy",
  "seed": 0,
  "folds_csv": "/kaggle/working/splits/folds.csv"
 },
 {
  "prefix": "s2_efficientnet_b3_s256_hv_seed1",
  "bb": "efficientnet_b3",
  "size": 256,
  "lr": 0.0003,
  "aug": "heavy",
  "seed": 1,
  "folds_csv": "/kaggle/working/splits/folds.csv"
 },
 {
  "prefix": "s2_efficientnetv2_s_s256_seed0",
  "bb": "efficientnetv2_s",
  "size": 256,
  "lr": 0.0003,
  "aug": "light",
  "seed": 0,
  "folds_csv": "/kaggle/working/splits/folds.csv"
 },
 {
  "prefix": "s2_densenet121_s256_seed0",
  "bb": "densenet121",
  "size": 256,
  "lr": 0.0003,
  "aug": "light",
  "seed": 0,
  "folds_csv": "/kaggle/working/splits/folds.csv"
 }
]
FOLDS = [0, 1, 2, 3, 4]
OUT = "/kaggle/working/runs"
CV = "/kaggle/working/splits/cv_folds.csv"
HOLD = "/kaggle/working/splits/holdout_uids.csv"

jobs = []
for r, f in itertools.product(RUNS, FOLDS):          # config-major order: a cut-off still leaves COMPLETE configs
    name = f"{r['prefix']}_f{f}"
    if os.path.exists(f"{OUT}/{name}/preds.npz"):
        continue
    cmd = ["python", "-m", "src.train_v2", "--backbone", r["bb"], "--img_size", str(r["size"]), "--aug", r.get("aug", "light"),
           "--lr", str(r["lr"]), "--epochs", str(r.get("epochs", EPOCHS)), "--batch_size", str(BATCH), "--fold", str(f),
           "--folds_csv", r.get("folds_csv", CV), "--fold_col", "fold", "--run_name", name, "--seed", str(r.get("seed", 42)),
           "--lam_ce", str(r.get("lam_ce", 0.5))]
    pass
    if r.get("pseudo_csv"): cmd += ["--pseudo_csv", r["pseudo_csv"]]
    jobs.append((name, cmd))
print(len(jobs), "jobs to run")

q = queue.Queue()
for j in jobs: q.put(j)
failed = []

def worker(gpu):
    while True:
        try: name, cmd = q.get_nowait()
        except queue.Empty: return
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        os.makedirs("/kaggle/working/logs", exist_ok=True)
        with open(f"/kaggle/working/logs/{name}.log", "w") as log:
            rc = subprocess.call(cmd, cwd="/kaggle/working", env=env, stdout=log, stderr=subprocess.STDOUT)
        print(("[ok]   " if rc == 0 else "[FAIL] ") + name, time.strftime("%H:%M:%S"))
        if rc != 0: failed.append(name)

ths = [threading.Thread(target=worker, args=(g,)) for g in range(max(1, torch.cuda.device_count()))]
[t.start() for t in ths]; [t.join() for t in ths]
print("failed:", failed)

In [ ]:
# keep model weights only for the runs needed later (cross-dataset evaluation); predictions (preds.npz) are always kept
import glob, os
KEEP = []
freed = 0
for p in glob.glob("/kaggle/working/runs/*/final_ema.pt"):
    if not any(os.path.basename(os.path.dirname(p)).startswith(k) for k in KEEP):
        freed += os.path.getsize(p); os.remove(p)
print(f"removed {freed / 1e9:.2f} GB of unneeded weights")

## 2a ensemble (OOF-only decisions) -> submission_s2a.csv

In [ ]:
import os
_runs = ['s2_convnext_tiny_s256_seed0', 's2_convnext_tiny_s256_seed1', 's2_resnet50_s256_heavy_seed0', 's2_resnet50_s256_heavy_seed1', 's2_efficientnet_b3_s256_hv_seed0', 's2_efficientnet_b3_s256_hv_seed1', 's2_efficientnetv2_s_s256_seed0', 's2_densenet121_s256_seed0']
done = [p for p in _runs if all(os.path.exists(f"/kaggle/working/runs/{p}_f{k}/preds.npz") for k in range(5))]
print("complete runs:", len(done), "of", len(_runs)); assert done, "no complete run -- see /kaggle/working/logs"
prefixes = ",".join(done)
!cd /kaggle/working && python -m src.run_ensemble --out_dir runs --runs {prefixes} --folds 0,1,2,3,4 --index_csv index.csv --cv_folds_csv splits/folds.csv --sample_submission {IMAGES_DATASET}/sample_submission.csv --result_dir results_s2a
!cp /kaggle/working/results_s2a/submission.csv /kaggle/working/submission_s2a.csv
import json; r = json.load(open("/kaggle/working/results_s2a/results.json"))
print("OOF (ensemble @0.5 + min-one):", {k: round(v, 4) for k, v in r["oof"]["ens_0.5_min_one"].items()})
print("weights:", {k: round(v, 3) for k, v in r["weights"].items() if v > 0})

## Confident pseudo-labels for the unlabelled test images (rules: pseudo-labelling is allowed)

In [ ]:
MIN_CONF = 0.9        # lower (0.8) keeps more images but noisier labels
!cd /kaggle/working && python -m src.make_pseudo_labels --ensemble_arrays results_s2a/ensemble_arrays.npz --index_csv index.csv --out splits/pseudo.csv --min_conf {MIN_CONF}

## 2b -- same configs, plus the pseudo-labelled test images in every training fold

In [ ]:
import subprocess, itertools, threading, queue, os, time

EPOCHS, BATCH = 30, 32
RUNS = [
 {
  "prefix": "s2p_convnext_tiny_s256_seed0",
  "bb": "convnext_tiny",
  "size": 256,
  "lr": 0.0001,
  "aug": "light",
  "seed": 0,
  "folds_csv": "/kaggle/working/splits/folds.csv",
  "pseudo_csv": "/kaggle/working/splits/pseudo.csv"
 },
 {
  "prefix": "s2p_convnext_tiny_s256_seed1",
  "bb": "convnext_tiny",
  "size": 256,
  "lr": 0.0001,
  "aug": "light",
  "seed": 1,
  "folds_csv": "/kaggle/working/splits/folds.csv",
  "pseudo_csv": "/kaggle/working/splits/pseudo.csv"
 },
 {
  "prefix": "s2p_resnet50_s256_heavy_seed0",
  "bb": "resnet50",
  "size": 256,
  "lr": 0.0003,
  "aug": "heavy",
  "seed": 0,
  "folds_csv": "/kaggle/working/splits/folds.csv",
  "pseudo_csv": "/kaggle/working/splits/pseudo.csv"
 },
 {
  "prefix": "s2p_resnet50_s256_heavy_seed1",
  "bb": "resnet50",
  "size": 256,
  "lr": 0.0003,
  "aug": "heavy",
  "seed": 1,
  "folds_csv": "/kaggle/working/splits/folds.csv",
  "pseudo_csv": "/kaggle/working/splits/pseudo.csv"
 },
 {
  "prefix": "s2p_efficientnet_b3_s256_hv_seed0",
  "bb": "efficientnet_b3",
  "size": 256,
  "lr": 0.0003,
  "aug": "heavy",
  "seed": 0,
  "folds_csv": "/kaggle/working/splits/folds.csv",
  "pseudo_csv": "/kaggle/working/splits/pseudo.csv"
 },
 {
  "prefix": "s2p_efficientnet_b3_s256_hv_seed1",
  "bb": "efficientnet_b3",
  "size": 256,
  "lr": 0.0003,
  "aug": "heavy",
  "seed": 1,
  "folds_csv": "/kaggle/working/splits/folds.csv",
  "pseudo_csv": "/kaggle/working/splits/pseudo.csv"
 },
 {
  "prefix": "s2p_efficientnetv2_s_s256_seed0",
  "bb": "efficientnetv2_s",
  "size": 256,
  "lr": 0.0003,
  "aug": "light",
  "seed": 0,
  "folds_csv": "/kaggle/working/splits/folds.csv",
  "pseudo_csv": "/kaggle/working/splits/pseudo.csv"
 },
 {
  "prefix": "s2p_densenet121_s256_seed0",
  "bb": "densenet121",
  "size": 256,
  "lr": 0.0003,
  "aug": "light",
  "seed": 0,
  "folds_csv": "/kaggle/working/splits/folds.csv",
  "pseudo_csv": "/kaggle/working/splits/pseudo.csv"
 }
]
FOLDS = [0, 1, 2, 3, 4]
OUT = "/kaggle/working/runs"
CV = "/kaggle/working/splits/cv_folds.csv"
HOLD = "/kaggle/working/splits/holdout_uids.csv"

jobs = []
for r, f in itertools.product(RUNS, FOLDS):          # config-major order: a cut-off still leaves COMPLETE configs
    name = f"{r['prefix']}_f{f}"
    if os.path.exists(f"{OUT}/{name}/preds.npz"):
        continue
    cmd = ["python", "-m", "src.train_v2", "--backbone", r["bb"], "--img_size", str(r["size"]), "--aug", r.get("aug", "light"),
           "--lr", str(r["lr"]), "--epochs", str(r.get("epochs", EPOCHS)), "--batch_size", str(BATCH), "--fold", str(f),
           "--folds_csv", r.get("folds_csv", CV), "--fold_col", "fold", "--run_name", name, "--seed", str(r.get("seed", 42)),
           "--lam_ce", str(r.get("lam_ce", 0.5))]
    pass
    if r.get("pseudo_csv"): cmd += ["--pseudo_csv", r["pseudo_csv"]]
    jobs.append((name, cmd))
print(len(jobs), "jobs to run")

q = queue.Queue()
for j in jobs: q.put(j)
failed = []

def worker(gpu):
    while True:
        try: name, cmd = q.get_nowait()
        except queue.Empty: return
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        os.makedirs("/kaggle/working/logs", exist_ok=True)
        with open(f"/kaggle/working/logs/{name}.log", "w") as log:
            rc = subprocess.call(cmd, cwd="/kaggle/working", env=env, stdout=log, stderr=subprocess.STDOUT)
        print(("[ok]   " if rc == 0 else "[FAIL] ") + name, time.strftime("%H:%M:%S"))
        if rc != 0: failed.append(name)

ths = [threading.Thread(target=worker, args=(g,)) for g in range(max(1, torch.cuda.device_count()))]
[t.start() for t in ths]; [t.join() for t in ths]
print("failed:", failed)

In [ ]:
# keep model weights only for the runs needed later (cross-dataset evaluation); predictions (preds.npz) are always kept
import glob, os
KEEP = []
freed = 0
for p in glob.glob("/kaggle/working/runs/*/final_ema.pt"):
    if not any(os.path.basename(os.path.dirname(p)).startswith(k) for k in KEEP):
        freed += os.path.getsize(p); os.remove(p)
print(f"removed {freed / 1e9:.2f} GB of unneeded weights")

## 2b ensemble -> submission_s2b.csv, and 2a+2b -> submission_s2u.csv

In [ ]:
import os
_runs = ['s2p_convnext_tiny_s256_seed0', 's2p_convnext_tiny_s256_seed1', 's2p_resnet50_s256_heavy_seed0', 's2p_resnet50_s256_heavy_seed1', 's2p_efficientnet_b3_s256_hv_seed0', 's2p_efficientnet_b3_s256_hv_seed1', 's2p_efficientnetv2_s_s256_seed0', 's2p_densenet121_s256_seed0']
done = [p for p in _runs if all(os.path.exists(f"/kaggle/working/runs/{p}_f{k}/preds.npz") for k in range(5))]
print("complete runs:", len(done), "of", len(_runs)); assert done, "no complete run -- see /kaggle/working/logs"
prefixes = ",".join(done)
!cd /kaggle/working && python -m src.run_ensemble --out_dir runs --runs {prefixes} --folds 0,1,2,3,4 --index_csv index.csv --cv_folds_csv splits/folds.csv --sample_submission {IMAGES_DATASET}/sample_submission.csv --result_dir results_s2b
!cp /kaggle/working/results_s2b/submission.csv /kaggle/working/submission_s2b.csv
import json; r = json.load(open("/kaggle/working/results_s2b/results.json"))
print("OOF (ensemble @0.5 + min-one):", {k: round(v, 4) for k, v in r["oof"]["ens_0.5_min_one"].items()})
print("weights:", {k: round(v, 3) for k, v in r["weights"].items() if v > 0})

In [ ]:
import os
_runs = ['s2_convnext_tiny_s256_seed0', 's2_convnext_tiny_s256_seed1', 's2_resnet50_s256_heavy_seed0', 's2_resnet50_s256_heavy_seed1', 's2_efficientnet_b3_s256_hv_seed0', 's2_efficientnet_b3_s256_hv_seed1', 's2_efficientnetv2_s_s256_seed0', 's2_densenet121_s256_seed0', 's2p_convnext_tiny_s256_seed0', 's2p_convnext_tiny_s256_seed1', 's2p_resnet50_s256_heavy_seed0', 's2p_resnet50_s256_heavy_seed1', 's2p_efficientnet_b3_s256_hv_seed0', 's2p_efficientnet_b3_s256_hv_seed1', 's2p_efficientnetv2_s_s256_seed0', 's2p_densenet121_s256_seed0']
done = [p for p in _runs if all(os.path.exists(f"/kaggle/working/runs/{p}_f{k}/preds.npz") for k in range(5))]
print("complete runs:", len(done), "of", len(_runs)); assert done, "no complete run -- see /kaggle/working/logs"
prefixes = ",".join(done)
!cd /kaggle/working && python -m src.run_ensemble --out_dir runs --runs {prefixes} --folds 0,1,2,3,4 --index_csv index.csv --cv_folds_csv splits/folds.csv --sample_submission {IMAGES_DATASET}/sample_submission.csv --result_dir results_s2u
!cp /kaggle/working/results_s2u/submission.csv /kaggle/working/submission_s2u.csv
import json; r = json.load(open("/kaggle/working/results_s2u/results.json"))
print("OOF (ensemble @0.5 + min-one):", {k: round(v, 4) for k, v in r["oof"]["ens_0.5_min_one"].items()})
print("weights:", {k: round(v, 3) for k, v in r["weights"].items() if v > 0})

## Next

Download `submission_s2a.csv`, `submission_s2b.csv`, `submission_s2u.csv` and the three `results_*` folders. Late-submit them (ONE account) and send me the private/public scores. Stage-1 reference: private 0.92563 / public 0.92704; competition #1 = 0.93378.